# Step 5Q - Hybrid QAOA Active Set

Executed project evidence and reproducible code.

## Write the Qiskit hybrid optimization module

In [5]:
%%writefile step_05q_hybrid_qaoa_final.py
from __future__ import annotations
from dataclasses import dataclass, replace
from itertools import combinations
import inspect
import json
import math
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
from step_05_tunable_goals_final import GoalMixConfig, GoalPreferences, GoalScales, Step5Context, exact_goal_components, solve_goal_profile
EPS = 1e-12

def as_numpy(values: Any) -> np.ndarray:
    if hasattr(values, 'to_numpy'):
        return values.to_numpy(dtype=float)
    return np.asarray(values, dtype=float)

def unit_interval(values: Any, higher_is_better: bool=True) -> np.ndarray:
    array = np.asarray(values, dtype=float)
    if not np.all(np.isfinite(array)):
        raise ValueError('All screening values must be finite.')
    lo = float(array.min())
    hi = float(array.max())
    if hi - lo <= EPS:
        score = np.full_like(array, 0.5, dtype=float)
    else:
        score = (array - lo) / (hi - lo)
    return score if higher_is_better else 1.0 - score

@dataclass(frozen=True)
class QiskitHybridConfig:
    max_qubits: int = 14
    cardinality: int | None = None
    reps: int = 2
    shots: int = 4096
    maxiter: int = 100
    seed: int = 12345
    top_quantum_samples: int = 30
    maximum_subsets_to_refine: int = 20
    material_incumbent_weight: float = 0.015
    scenario_proxy_share: float = 0.35
    class_balance_strength: float = 0.1
    selected_minimum_weight: float = 0.0
    exact_enumeration_limit: int = 10000
    run_numpy_exact_eigensolver: bool = False
    retain_raw_qiskit_objects: bool = False
    initial_point: tuple[float, ...] | None = None
    callback_checkpoint_path: str | None = None
    callback_checkpoint_interval: int = 2
    selection_mode: str = 'active_rebalance'
    selection_objective: str = 'incumbent_marginal_utility'
    active_balance_strength: float = 1.0
    trade_materiality_floor: float = 0.0005
    trade_materiality_fraction: float = 0.01
    marginal_transfer_size: float = 0.0025
    marginal_improvement_fraction: float = 0.05
    inferred_cardinality_cap: int | None = 6
    qaoa_aggregation: float | None = 0.25
    transpiler_optimization_level: int = 2
    use_cardinality_preserving_mixer: bool = True
    exact_candidates_to_refine: int = 12
    executed_trade_threshold: float = 1e-05

    def validate(self) -> None:
        if not 4 <= self.max_qubits <= 20:
            raise ValueError('max_qubits must lie between 4 and 20.')
        if self.cardinality is not None and self.cardinality < 2:
            raise ValueError('cardinality must be at least two.')
        if self.reps < 1 or self.shots < 1 or self.maxiter < 1:
            raise ValueError('reps, shots, and maxiter must be positive.')
        if self.top_quantum_samples < 1:
            raise ValueError('top_quantum_samples must be positive.')
        if self.maximum_subsets_to_refine < 1:
            raise ValueError('maximum_subsets_to_refine must be positive.')
        if self.callback_checkpoint_interval < 1:
            raise ValueError('callback_checkpoint_interval must be positive.')
        if self.initial_point is not None:
            expected = 2 * self.reps
            if len(self.initial_point) != expected:
                raise ValueError(f'initial_point must contain {expected} values for reps={self.reps}.')
        if self.selection_mode not in {'active_rebalance', 'whole_support'}:
            raise ValueError("selection_mode must be 'active_rebalance' or 'whole_support'.")
        if self.selection_objective not in {'classical_target_recovery', 'incumbent_marginal_utility'}:
            raise ValueError("selection_objective must be 'classical_target_recovery' or 'incumbent_marginal_utility'.")
        if self.active_balance_strength < 0.0:
            raise ValueError('active_balance_strength must be nonnegative.')
        if not 0.0 <= self.scenario_proxy_share <= 1.0:
            raise ValueError('scenario_proxy_share must lie in [0, 1].')
        if self.class_balance_strength < 0.0:
            raise ValueError('class_balance_strength must be nonnegative.')
        if self.trade_materiality_floor < 0.0:
            raise ValueError('trade_materiality_floor must be nonnegative.')
        if not 0.0 <= self.trade_materiality_fraction <= 1.0:
            raise ValueError('trade_materiality_fraction must lie in [0, 1].')
        if self.marginal_transfer_size <= 0.0:
            raise ValueError('marginal_transfer_size must be positive.')
        if not 0.0 <= self.marginal_improvement_fraction <= 1.0:
            raise ValueError('marginal_improvement_fraction must lie in [0, 1].')
        if self.inferred_cardinality_cap is not None and self.inferred_cardinality_cap < 2:
            raise ValueError('inferred_cardinality_cap must be at least two.')
        if self.qaoa_aggregation is not None and (not 0.0 < self.qaoa_aggregation <= 1.0):
            raise ValueError('qaoa_aggregation must lie in (0, 1].')
        if self.transpiler_optimization_level not in {0, 1, 2, 3}:
            raise ValueError('transpiler_optimization_level must be 0, 1, 2, or 3.')
        if self.exact_candidates_to_refine < 1:
            raise ValueError('exact_candidates_to_refine must be positive.')
        if self.executed_trade_threshold < 0.0:
            raise ValueError('executed_trade_threshold must be nonnegative.')

@dataclass
class ReducedUniverse:
    tickers: list[str]
    full_indices: np.ndarray
    screening_table: pd.DataFrame
    required_class_counts: dict[str, int]
    minimum_feasible_cardinality: int

    @property
    def n_qubits(self) -> int:
        return len(self.tickers)

@dataclass
class BinarySelectionModel:
    tickers: list[str]
    Q: np.ndarray
    linear: np.ndarray
    constant: float
    cardinality: int
    target_class_counts: dict[str, int]

    @property
    def n_variables(self) -> int:
        return len(self.tickers)

    def energy(self, bits: Any) -> float:
        z = np.asarray(bits, dtype=float)
        return float(z @ self.Q @ z + self.linear @ z + self.constant)

def minimum_assets_for_weight(upper_bounds: Any, required_weight: float) -> int:
    if required_weight <= EPS:
        return 0
    caps = np.sort(np.asarray(upper_bounds, dtype=float))[::-1]
    feasible = np.flatnonzero(np.cumsum(caps) >= required_weight - 1e-12)
    if feasible.size == 0:
        raise ValueError(f'Asset caps cannot support required weight {required_weight:.4f}.')
    return int(feasible[0] + 1)

def required_class_counts(asset_classes: Any, asset_upper: Any, class_bounds: dict[str, tuple[float, float]]) -> dict[str, int]:
    labels = np.asarray(asset_classes, dtype=object)
    upper = np.asarray(asset_upper, dtype=float)
    result: dict[str, int] = {}
    for asset_class, (lower, _) in class_bounds.items():
        if lower <= EPS:
            continue
        mask = labels == asset_class
        if not mask.any():
            raise ValueError(f'No asset is available for required class {asset_class}.')
        result[asset_class] = minimum_assets_for_weight(upper[mask], float(lower))
    return result

def build_screening_table(*, context: Step5Context, preferences: GoalPreferences) -> pd.DataFrame:
    data = context.portfolio_data
    shares = preferences.shares
    w0 = as_numpy(context.current_weights)
    covariance = np.asarray(data.covariance, dtype=float)
    standalone_variance = np.diag(covariance)
    losses = np.asarray(context.scenarios.loss_matrix, dtype=float)
    scenario_weights = np.asarray(context.scenarios.weights, dtype=float)
    scenario_rms = np.sqrt(np.average(losses ** 2, axis=0, weights=scenario_weights))
    impact = np.asarray(data.impact_matrix, dtype=float)
    impact_diagonal = np.diag(impact) if impact.ndim == 2 else impact
    implementation_burden = np.asarray(data.linear_cost, dtype=float) + impact_diagonal
    growth_score = unit_interval(data.growth, True)
    income_score = unit_interval(data.income, True)
    variance_score = unit_interval(standalone_variance, False)
    stress_score = unit_interval(scenario_rms, False)
    cost_score = unit_interval(implementation_burden, False)
    incumbent_score = unit_interval(w0, True)
    screening_score = shares['growth'] * growth_score + shares['income'] * income_score + shares['drawdown'] * (0.5 * variance_score + 0.5 * stress_score) + shares['cost'] * (0.6 * cost_score + 0.4 * incumbent_score)
    return pd.DataFrame({'ticker': list(data.tickers), 'asset_class': list(data.asset_classes), 'current_weight': w0, 'growth': np.asarray(data.growth, dtype=float), 'income': np.asarray(data.income, dtype=float), 'standalone_variance': standalone_variance, 'scenario_rms_loss': scenario_rms, 'linear_cost': np.asarray(data.linear_cost, dtype=float), 'impact_diagonal': impact_diagonal, 'screening_score': screening_score}).set_index('ticker')

def reduce_universe(*, context: Step5Context, preferences: GoalPreferences, config: QiskitHybridConfig) -> ReducedUniverse:
    config.validate()
    data = context.portfolio_data
    constraints = context.constraints
    table = build_screening_table(context=context, preferences=preferences)
    labels = np.asarray(data.asset_classes, dtype=object)
    upper = np.asarray(constraints.asset_upper, dtype=float)
    class_counts = required_class_counts(labels, upper, constraints.class_bounds)
    global_count = minimum_assets_for_weight(upper, 1.0)
    minimum_cardinality = max(global_count, int(sum(class_counts.values())))
    if config.max_qubits < minimum_cardinality:
        raise ValueError(f'At least {minimum_cardinality} selected assets are needed, but max_qubits={config.max_qubits}.')
    selected: list[str] = []

    def add(ticker: str) -> None:
        if ticker not in selected and len(selected) < config.max_qubits:
            selected.append(ticker)
    for asset_class, count in class_counts.items():
        ranked = table.loc[table['asset_class'] == asset_class].sort_values('screening_score', ascending=False)
        for ticker in ranked.head(count).index:
            add(str(ticker))
    incumbent_ranked = table.loc[table['current_weight'] >= config.material_incumbent_weight].sort_values('current_weight', ascending=False)
    for ticker in incumbent_ranked.index:
        add(str(ticker))
    for ticker in table.sort_values('screening_score', ascending=False).index:
        add(str(ticker))
    if len(selected) != config.max_qubits:
        raise RuntimeError('Could not construct the requested qubit universe.')
    full_tickers = list(data.tickers)
    indices = np.asarray([full_tickers.index(ticker) for ticker in selected], dtype=int)
    selected_table = table.loc[selected].copy()
    selected_table['qubit_index'] = np.arange(len(selected))
    return ReducedUniverse(tickers=selected, full_indices=indices, screening_table=selected_table, required_class_counts=class_counts, minimum_feasible_cardinality=minimum_cardinality)

def derive_step5_coefficients(preferences: GoalPreferences, scales: GoalScales, mix: GoalMixConfig) -> dict[str, float]:
    preferences.validate()
    scales.validate()
    mix.validate()
    shares = preferences.shares
    return {'growth': shares['growth'] / scales.growth, 'income': shares['income'] / scales.income, 'variance': shares['drawdown'] * mix.variance_share_of_drawdown / scales.variance, 'scenario': (shares['drawdown'] * mix.scenario_share_of_drawdown + mix.scenario_tiebreaker) / scales.scenario_hinge, 'linear_cost': (shares['cost'] * mix.linear_cost_share + mix.execution_tiebreaker) / scales.linear_cost, 'impact': (shares['cost'] * mix.impact_cost_share + mix.execution_tiebreaker) / scales.impact_cost, 'turnover': (shares['cost'] * mix.turnover_share + mix.execution_tiebreaker) / scales.turnover, 'concentration': mix.concentration_tiebreaker / scales.concentration}

def largest_remainder_class_targets(*, labels: np.ndarray, required_counts: dict[str, int], cardinality: int) -> dict[str, int]:
    classes = sorted(set(labels.tolist()))
    available = {asset_class: int(np.sum(labels == asset_class)) for asset_class in classes}
    targets = {asset_class: min(required_counts.get(asset_class, 0), available[asset_class]) for asset_class in classes}
    remaining = cardinality - sum(targets.values())
    if remaining < 0:
        raise ValueError('Required class support exceeds selected cardinality.')
    availability = np.asarray([available[asset_class] for asset_class in classes], dtype=float)
    availability /= availability.sum()
    desired = remaining * availability
    floors = np.floor(desired).astype(int)
    for asset_class, extra in zip(classes, floors, strict=True):
        capacity = available[asset_class] - targets[asset_class]
        targets[asset_class] += min(int(extra), capacity)
    remainder = cardinality - sum(targets.values())
    fractional = desired - floors
    while remainder > 0:
        feasible = [index for index, asset_class in enumerate(classes) if targets[asset_class] < available[asset_class]]
        if not feasible:
            raise ValueError('Reduced universe cannot support the chosen cardinality.')
        best = max(feasible, key=lambda index: fractional[index])
        targets[classes[best]] += 1
        fractional[best] = -np.inf
        remainder -= 1
    return targets

def build_binary_selection_model(*, context: Step5Context, reduced: ReducedUniverse, preferences: GoalPreferences, scales: GoalScales, mix: GoalMixConfig, cardinality: int, config: QiskitHybridConfig) -> BinarySelectionModel:
    data = context.portfolio_data
    idx = reduced.full_indices
    m = reduced.n_qubits
    k = int(cardinality)
    proxy_weight = 1.0 / k
    coeff = derive_step5_coefficients(preferences, scales, mix)
    growth = np.asarray(data.growth, dtype=float)[idx]
    income = np.asarray(data.income, dtype=float)[idx]
    covariance = np.asarray(data.covariance, dtype=float)[np.ix_(idx, idx)]
    linear_cost = np.asarray(data.linear_cost, dtype=float)[idx]
    impact = np.asarray(data.impact_matrix, dtype=float)[np.ix_(idx, idx)]
    incumbent = as_numpy(context.current_weights)[idx]
    loss_matrix = np.asarray(context.scenarios.loss_matrix, dtype=float)[:, idx]
    scenario_weights = np.asarray(context.scenarios.weights, dtype=float)
    Q = coeff['variance'] * proxy_weight ** 2 * covariance + config.scenario_proxy_share * coeff['scenario'] * proxy_weight ** 2 * (loss_matrix.T @ np.diag(scenario_weights) @ loss_matrix) + coeff['impact'] * proxy_weight ** 2 * impact + coeff['concentration'] * proxy_weight ** 2 * np.eye(m)
    linear = -coeff['growth'] * proxy_weight * growth - coeff['income'] * proxy_weight * income
    constant = 0.0
    selected_delta = np.abs(proxy_weight - incumbent)
    unselected_delta = incumbent
    linear += coeff['linear_cost'] * linear_cost * (selected_delta - unselected_delta)
    constant += float(coeff['linear_cost'] * linear_cost @ unselected_delta)
    linear += coeff['turnover'] * (selected_delta - unselected_delta)
    constant += float(coeff['turnover'] * unselected_delta.sum())
    linear += -2.0 * coeff['impact'] * proxy_weight * (impact @ incumbent)
    constant += float(coeff['impact'] * incumbent @ impact @ incumbent)
    labels = np.asarray(data.asset_classes, dtype=object)[idx]
    class_targets = largest_remainder_class_targets(labels=labels, required_counts=reduced.required_class_counts, cardinality=k)
    if config.class_balance_strength > 0.0:
        magnitude = max(float(np.max(np.abs(Q))), float(np.max(np.abs(linear))), 0.001)
        penalty = config.class_balance_strength * magnitude
        for asset_class, target in class_targets.items():
            indicator = (labels == asset_class).astype(float)
            Q += penalty * np.outer(indicator, indicator)
            linear += -2.0 * penalty * target * indicator
            constant += penalty * target ** 2
    Q = 0.5 * (Q + Q.T)
    return BinarySelectionModel(tickers=list(reduced.tickers), Q=Q, linear=linear, constant=float(constant), cardinality=k, target_class_counts=class_targets)

def build_qiskit_quadratic_program(model: BinarySelectionModel) -> Any:
    from qiskit_optimization.problems import QuadraticProgram
    qp = QuadraticProgram('hybrid_portfolio_selection')
    for ticker in model.tickers:
        qp.binary_var(name=ticker)
    linear = {ticker: float(model.linear[i] + model.Q[i, i]) for i, ticker in enumerate(model.tickers)}
    quadratic: dict[tuple[str, str], float] = {}
    for i in range(model.n_variables):
        for j in range(i + 1, model.n_variables):
            coefficient = float(2.0 * model.Q[i, j])
            if abs(coefficient) > 1e-15:
                quadratic[model.tickers[i], model.tickers[j]] = coefficient
    qp.minimize(constant=model.constant, linear=linear, quadratic=quadratic)
    qp.linear_constraint(linear={ticker: 1.0 for ticker in model.tickers}, sense='==', rhs=float(model.cardinality), name='fixed_cardinality')
    return qp

def enumerate_fixed_cardinality(model: BinarySelectionModel, maximum_states: int) -> pd.DataFrame:
    number = math.comb(model.n_variables, model.cardinality)
    if number > maximum_states:
        return pd.DataFrame()
    records: list[dict[str, Any]] = []
    for chosen in combinations(range(model.n_variables), model.cardinality):
        bits = np.zeros(model.n_variables, dtype=int)
        bits[list(chosen)] = 1
        records.append({'bitstring': ''.join((str(int(value)) for value in bits)), 'energy': model.energy(bits), 'selected_indices': tuple(chosen), 'selected_tickers': ', '.join((model.tickers[index] for index in chosen))})
    return pd.DataFrame(records).sort_values('energy', ascending=True).reset_index(drop=True)

def build_fixed_cardinality_qaoa_components(*, n_qubits: int, cardinality: int) -> tuple[Any, Any]:
    from qiskit import QuantumCircuit
    from qiskit.quantum_info import SparsePauliOp
    if not 0 < cardinality < n_qubits:
        raise ValueError('cardinality must lie strictly between zero and n_qubits.')
    amplitudes = np.zeros(2 ** n_qubits, dtype=complex)
    normalization = math.sqrt(math.comb(n_qubits, cardinality))
    for chosen in combinations(range(n_qubits), cardinality):
        basis_index = sum((1 << qubit for qubit in chosen))
        amplitudes[basis_index] = 1.0 / normalization
    initial_state = QuantumCircuit(n_qubits)
    initial_state.initialize(amplitudes, range(n_qubits))
    paulis: list[str] = []
    coefficients: list[float] = []
    for left in range(n_qubits):
        right = (left + 1) % n_qubits
        for symbol in ('X', 'Y'):
            label = ['I'] * n_qubits
            label[n_qubits - 1 - left] = symbol
            label[n_qubits - 1 - right] = symbol
            paulis.append(''.join(label))
            coefficients.append(0.5)
    mixer = SparsePauliOp(paulis, coeffs=coefficients)
    return (initial_state, mixer)

def run_qiskit_qaoa(*, quadratic_program: Any, model: BinarySelectionModel, config: QiskitHybridConfig) -> dict[str, Any]:
    from qiskit_aer import AerSimulator
    from qiskit_aer.primitives import SamplerV2 as AerSamplerV2
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
    from qiskit_optimization.algorithms import MinimumEigenOptimizer
    try:
        from qiskit_optimization.minimum_eigensolvers import QAOA
        from qiskit_optimization.optimizers import COBYLA
        qaoa_api_family = 'qiskit_optimization'
    except ImportError:
        from qiskit_algorithms import QAOA
        from qiskit_algorithms.optimizers import COBYLA
        qaoa_api_family = 'qiskit_algorithms'
    callback_rows: list[dict[str, Any]] = []
    best_callback_state: dict[str, Any] = {'mean_energy': float('inf'), 'evaluation': 0, 'parameters': None, 'metadata': None}

    def _json_safe(value: Any) -> Any:
        if value is None or isinstance(value, (str, bool, int, float)):
            return value
        if isinstance(value, np.generic):
            return value.item()
        if isinstance(value, np.ndarray):
            return value.tolist()
        if isinstance(value, dict):
            return {str(key): _json_safe(item) for key, item in value.items()}
        if isinstance(value, (list, tuple)):
            return [_json_safe(item) for item in value]
        return repr(value)

    def _metadata_standard_deviation(metadata: Any) -> float:
        if not isinstance(metadata, dict):
            try:
                return float(metadata)
            except (TypeError, ValueError):
                return float('nan')
        for key in ('standard_deviation', 'stddev', 'std'):
            value = metadata.get(key)
            if np.isscalar(value) and value is not None:
                try:
                    return float(value)
                except (TypeError, ValueError):
                    pass
        for key in ('variance', 'variance_estimate'):
            value = metadata.get(key)
            if np.isscalar(value) and value is not None:
                try:
                    return float(np.sqrt(max(float(value), 0.0)))
                except (TypeError, ValueError):
                    pass
        return float('nan')

    def callback(evaluation_count: int, parameters: np.ndarray, mean: float, metadata: dict[str, Any]) -> None:
        safe_metadata = _json_safe(metadata)
        parameter_list = np.asarray(parameters, dtype=float).tolist()
        mean_value = float(np.real(mean))
        callback_rows.append({'evaluation': int(evaluation_count), 'mean_energy': mean_value, 'standard_deviation': _metadata_standard_deviation(metadata), 'parameters': parameter_list, 'metadata': safe_metadata, 'metadata_json': json.dumps(safe_metadata, sort_keys=True)})
        if mean_value < float(best_callback_state['mean_energy']):
            best_callback_state.update({'mean_energy': mean_value, 'evaluation': int(evaluation_count), 'parameters': parameter_list, 'metadata': safe_metadata})
        if config.callback_checkpoint_path and int(evaluation_count) % int(config.callback_checkpoint_interval) == 0:
            checkpoint = Path(config.callback_checkpoint_path)
            checkpoint.parent.mkdir(parents=True, exist_ok=True)
            temporary = checkpoint.with_suffix(checkpoint.suffix + '.tmp')
            temporary.write_text(json.dumps(best_callback_state, indent=2, sort_keys=True), encoding='utf-8')
            temporary.replace(checkpoint)
    aer_backend = AerSimulator(method='statevector', precision='single')
    sampler = AerSamplerV2(default_shots=config.shots, seed=config.seed, options={'backend_options': {'method': 'statevector', 'precision': 'single'}})
    qaoa_transpiler = generate_preset_pass_manager(optimization_level=config.transpiler_optimization_level, backend=aer_backend)
    if config.initial_point is None:
        initial_point = np.concatenate([np.full(config.reps, 0.5, dtype=float), np.full(config.reps, 0.5, dtype=float)])
    else:
        initial_point = np.asarray(config.initial_point, dtype=float)
    qaoa_kwargs: dict[str, Any] = {'sampler': sampler, 'optimizer': COBYLA(maxiter=config.maxiter), 'reps': config.reps, 'initial_point': initial_point, 'callback': callback}
    if config.qaoa_aggregation is not None:
        qaoa_kwargs['aggregation'] = config.qaoa_aggregation
    if config.use_cardinality_preserving_mixer:
        initial_state, mixer = build_fixed_cardinality_qaoa_components(n_qubits=model.n_variables, cardinality=model.cardinality)
        qaoa_kwargs['initial_state'] = initial_state
        qaoa_kwargs['mixer'] = mixer
        mixer_type = 'XY_ring_with_Dicke_initial_state'
    else:
        mixer_type = 'default_X_mixer'
    qaoa_signature = inspect.signature(QAOA)
    qaoa_parameters = qaoa_signature.parameters
    if 'pass_manager' in qaoa_parameters:
        qaoa_kwargs['pass_manager'] = qaoa_transpiler
        transpiler_keyword = 'pass_manager'
    elif 'transpiler' in qaoa_parameters:
        qaoa_kwargs['transpiler'] = qaoa_transpiler
        transpiler_keyword = 'transpiler'
    else:
        raise RuntimeError('The installed QAOA constructor exposes neither `pass_manager` nor `transpiler`; its circuits cannot be prepared safely for Aer SamplerV2.')
    qaoa = QAOA(**qaoa_kwargs)
    qaoa_optimizer = MinimumEigenOptimizer(qaoa)
    qaoa_result = qaoa_optimizer.solve(quadratic_program)
    exact_result = None
    if config.run_numpy_exact_eigensolver:
        from qiskit_algorithms import NumPyMinimumEigensolver
        exact_optimizer = MinimumEigenOptimizer(NumPyMinimumEigensolver())
        exact_result = exact_optimizer.solve(quadratic_program)
    sample_records: list[dict[str, Any]] = []
    for sample in qaoa_result.samples:
        bits = np.asarray(sample.x, dtype=int)
        sample_records.append({'bitstring': ''.join((str(int(value)) for value in bits)), 'cardinality': int(bits.sum()), 'raw_solver_probability': float(sample.probability), 'reported_objective': float(sample.fval), 'economic_energy': model.energy(bits), 'status': str(sample.status), 'selected_indices': tuple(np.flatnonzero(bits).tolist()), 'selected_tickers': ', '.join((model.tickers[index] for index in np.flatnonzero(bits)))})
    samples = pd.DataFrame(sample_records)
    if samples.empty:
        raise RuntimeError('QAOA returned no interpreted samples.')
    samples = samples.loc[samples['cardinality'] == model.cardinality].copy()
    if samples.empty:
        raise RuntimeError('QAOA returned no fixed-cardinality samples. Enable the cardinality-preserving mixer or increase shots.')
    feasible_probability_mass = float(samples['raw_solver_probability'].sum())
    denominator = max(feasible_probability_mass, 1e-15)
    samples['conditional_probability'] = samples['raw_solver_probability'] / denominator
    samples['probability_is_near_uniform'] = samples['conditional_probability'].max() - samples['conditional_probability'].min() <= 1e-12
    samples = samples.sort_values(['economic_energy', 'conditional_probability'], ascending=[True, False]).head(config.top_quantum_samples).reset_index(drop=True)
    exact_bits = np.asarray(exact_result.x, dtype=int) if exact_result is not None else None
    compact_result = {'samples': samples, 'callback_history': pd.DataFrame(callback_rows), 'exact_bits': exact_bits, 'exact_energy': model.energy(exact_bits) if exact_bits is not None else np.nan, 'optimizer_time': float(getattr(qaoa_result.min_eigen_solver_result, 'optimizer_time', np.nan)), 'optimal_point': np.asarray(getattr(qaoa_result.min_eigen_solver_result, 'optimal_point', np.array([], dtype=float)), dtype=float), 'callback_checkpoint_path': config.callback_checkpoint_path, 'transpiler_keyword': transpiler_keyword, 'transpiler_optimization_level': config.transpiler_optimization_level, 'aer_method': 'statevector', 'qaoa_api_family': qaoa_api_family, 'mixer_type': mixer_type, 'aggregation': config.qaoa_aggregation, 'feasible_probability_mass': feasible_probability_mass, 'aer_precision': 'single', 'optimizer_evaluations': int(getattr(qaoa_result.min_eigen_solver_result, 'cost_function_evals', len(callback_rows)) or len(callback_rows))}
    if config.retain_raw_qiskit_objects:
        compact_result.update({'qaoa_object': qaoa, 'qaoa_result': qaoa_result, 'exact_result': exact_result})
    return compact_result

def repair_selection(*, selected_indices: Any, reduced: ReducedUniverse, context: Step5Context, cardinality: int) -> np.ndarray:
    selected = set((int(index) for index in selected_indices))
    labels = np.asarray(context.portfolio_data.asset_classes, dtype=object)[reduced.full_indices]
    scores = reduced.screening_table['screening_score'].to_numpy()
    ranked = list(np.argsort(scores)[::-1])
    for index in ranked:
        if len(selected) >= cardinality:
            break
        selected.add(int(index))
    while len(selected) > cardinality:
        outgoing = min(selected, key=lambda index: scores[index])
        selected.remove(outgoing)

    def class_count(asset_class: str) -> int:
        return sum((labels[index] == asset_class for index in selected))
    for asset_class, required in reduced.required_class_counts.items():
        while class_count(asset_class) < required:
            incoming_options = [index for index in range(reduced.n_qubits) if index not in selected and labels[index] == asset_class]
            if not incoming_options:
                raise ValueError(f'Cannot repair class coverage for {asset_class}.')
            incoming = max(incoming_options, key=lambda index: scores[index])
            removable = [index for index in selected if class_count(str(labels[index])) > reduced.required_class_counts.get(str(labels[index]), 0)]
            if not removable:
                raise ValueError('No removable asset remains during class repair.')
            outgoing = min(removable, key=lambda index: scores[index])
            selected.remove(outgoing)
            selected.add(incoming)
    bits = np.zeros(reduced.n_qubits, dtype=int)
    bits[list(selected)] = 1
    return bits

def clone_constraints_for_subset(*, constraints: Any, selected_full_mask: np.ndarray, selected_minimum_weight: float) -> Any:
    lower = np.asarray(constraints.asset_lower, dtype=float).copy()
    upper = np.asarray(constraints.asset_upper, dtype=float).copy()
    lower[~selected_full_mask] = 0.0
    upper[~selected_full_mask] = 0.0
    if selected_minimum_weight > 0.0:
        lower[selected_full_mask] = np.maximum(lower[selected_full_mask], selected_minimum_weight)
    if lower.sum() > 1.0 + 1e-10:
        raise ValueError('Selected minimum weights exceed full investment.')
    if upper.sum() < 1.0 - 1e-10:
        raise ValueError('Selected asset caps cannot support full investment.')
    return replace(constraints, asset_lower=lower, asset_upper=upper)

def project_warm_start(*, source_weights: Any, selected_full_mask: np.ndarray, lower: np.ndarray, upper: np.ndarray) -> np.ndarray:
    weights = as_numpy(source_weights).copy()
    weights[~selected_full_mask] = 0.0
    weights = np.maximum(weights, lower)
    weights = np.minimum(weights, upper)
    for _ in range(200):
        difference = 1.0 - float(weights.sum())
        if abs(difference) <= 1e-10:
            break
        if difference > 0.0:
            slack = np.maximum(upper - weights, 0.0)
        else:
            slack = np.maximum(weights - lower, 0.0)
        total_slack = float(slack.sum())
        if total_slack <= EPS:
            break
        weights += difference * slack / total_slack
        weights = np.maximum(weights, lower)
        weights = np.minimum(weights, upper)
    if abs(float(weights.sum()) - 1.0) > 1e-07:
        raise ValueError('Could not construct a feasible warm start for the subset.')
    return weights

def build_active_screening_table(*, context: Step5Context, classical_reference: dict[str, Any], preferences: GoalPreferences) -> pd.DataFrame:
    table = build_screening_table(context=context, preferences=preferences).copy()
    current = as_numpy(context.current_weights)
    reference = as_numpy(classical_reference['result'].weights)
    desired_trade = reference - current
    absolute_trade = np.abs(desired_trade)
    data = context.portfolio_data
    impact = np.asarray(data.impact_matrix, dtype=float)
    impact_diagonal = np.diag(impact) if impact.ndim == 2 else impact
    implementation_burden = np.asarray(data.linear_cost, dtype=float) + impact_diagonal
    max_trade = max(float(absolute_trade.max()), 1e-12)
    trade_materiality = absolute_trade / max_trade
    base_score = unit_interval(table['screening_score'].to_numpy(dtype=float), True)
    low_cost_score = unit_interval(implementation_burden, False)
    active_score = trade_materiality * (0.85 + 0.1 * base_score + 0.05 * low_cost_score)
    table['reference_weight'] = reference
    table['reference_trade'] = desired_trade
    table['absolute_reference_trade'] = absolute_trade
    table['marginal_direction'] = np.sign(desired_trade).astype(int)
    table['marginal_improvement'] = np.nan
    table['marginal_counterparty'] = ''
    table['selection_trade'] = desired_trade
    table['absolute_selection_trade'] = absolute_trade
    table['trade_materiality'] = trade_materiality
    table['implementation_burden'] = implementation_burden
    table['active_selection_score'] = active_score
    table['selection_objective'] = 'classical_target_recovery'
    return table

def build_marginal_utility_screening_table(*, context: Step5Context, preferences: GoalPreferences, scales: GoalScales, mix: GoalMixConfig, config: QiskitHybridConfig) -> pd.DataFrame:
    table = build_screening_table(context=context, preferences=preferences).copy()
    current = as_numpy(context.current_weights)
    lower = np.asarray(context.constraints.asset_lower, dtype=float)
    upper = np.asarray(context.constraints.asset_upper, dtype=float)
    n_assets = len(current)
    epsilon = float(config.marginal_transfer_size)
    base = normalized_step5_objective(context=context, weights=current, preferences=preferences, scales=scales, mix=mix)['normalized_objective']
    best_improvement = np.full(n_assets, -np.inf, dtype=float)
    best_direction = np.zeros(n_assets, dtype=int)
    best_counterparty = np.full(n_assets, -1, dtype=int)
    for asset in range(n_assets):
        if current[asset] + epsilon <= upper[asset] + 1e-12:
            for counterparty in range(n_assets):
                if counterparty == asset:
                    continue
                if current[counterparty] - epsilon < lower[counterparty] - 1e-12:
                    continue
                trial = current.copy()
                trial[asset] += epsilon
                trial[counterparty] -= epsilon
                value = normalized_step5_objective(context=context, weights=trial, preferences=preferences, scales=scales, mix=mix)['normalized_objective']
                improvement = float(base - value)
                if improvement > best_improvement[asset]:
                    best_improvement[asset] = improvement
                    best_direction[asset] = 1
                    best_counterparty[asset] = counterparty
        if current[asset] - epsilon >= lower[asset] - 1e-12:
            for counterparty in range(n_assets):
                if counterparty == asset:
                    continue
                if current[counterparty] + epsilon > upper[counterparty] + 1e-12:
                    continue
                trial = current.copy()
                trial[asset] -= epsilon
                trial[counterparty] += epsilon
                value = normalized_step5_objective(context=context, weights=trial, preferences=preferences, scales=scales, mix=mix)['normalized_objective']
                improvement = float(base - value)
                if improvement > best_improvement[asset]:
                    best_improvement[asset] = improvement
                    best_direction[asset] = -1
                    best_counterparty[asset] = counterparty
    finite = np.isfinite(best_improvement)
    best_improvement[~finite] = 0.0
    positive_improvement = np.maximum(best_improvement, 0.0)
    maximum_improvement = max(float(positive_improvement.max()), 1e-16)
    marginal_materiality = positive_improvement / maximum_improvement
    data = context.portfolio_data
    impact = np.asarray(data.impact_matrix, dtype=float)
    impact_diagonal = np.diag(impact) if impact.ndim == 2 else impact
    implementation_burden = np.asarray(data.linear_cost, dtype=float) + impact_diagonal
    low_cost_score = unit_interval(implementation_burden, False)
    base_score = unit_interval(table['screening_score'].to_numpy(dtype=float), True)
    active_score = marginal_materiality * (0.85 + 0.1 * base_score + 0.05 * low_cost_score)
    proxy_trade = best_direction.astype(float) * epsilon * marginal_materiality
    tickers = list(context.portfolio_data.tickers)
    counterparties = [tickers[index] if index >= 0 else '' for index in best_counterparty]
    table['reference_weight'] = np.nan
    table['reference_trade'] = np.nan
    table['absolute_reference_trade'] = np.nan
    table['marginal_direction'] = best_direction
    table['marginal_improvement'] = positive_improvement
    table['marginal_counterparty'] = counterparties
    table['selection_trade'] = proxy_trade
    table['absolute_selection_trade'] = np.abs(proxy_trade)
    table['trade_materiality'] = marginal_materiality
    table['implementation_burden'] = implementation_burden
    table['active_selection_score'] = active_score
    table['selection_objective'] = 'incumbent_marginal_utility'
    return table

def infer_active_cardinality(*, reduced: ReducedUniverse, config: QiskitHybridConfig) -> int:
    if config.cardinality is not None:
        cardinality = int(config.cardinality)
        if not 1 <= cardinality <= reduced.n_qubits:
            raise ValueError('Configured cardinality must be between 1 and the reduced-universe size.')
        return cardinality
    if config.selection_objective == 'classical_target_recovery':
        magnitude = reduced.screening_table['absolute_reference_trade'].to_numpy(dtype=float)
        maximum = max(float(np.nanmax(magnitude)), 0.0)
        threshold = max(float(config.trade_materiality_floor), float(config.trade_materiality_fraction) * maximum)
    elif config.selection_objective == 'incumbent_marginal_utility':
        magnitude = reduced.screening_table['marginal_improvement'].to_numpy(dtype=float)
        maximum = max(float(np.nanmax(magnitude)), 0.0)
        threshold = max(1e-12, float(config.marginal_improvement_fraction) * maximum)
    else:
        raise ValueError(f'Unknown selection objective: {config.selection_objective!r}')
    count = int(np.count_nonzero(np.isfinite(magnitude) & (magnitude >= threshold)))
    upper = max(2, reduced.n_qubits - 1)
    if config.inferred_cardinality_cap is not None:
        upper = min(upper, int(config.inferred_cardinality_cap))
    return int(np.clip(count, 2, upper))

def reduce_active_universe(*, context: Step5Context, classical_reference: dict[str, Any], preferences: GoalPreferences, scales: GoalScales, mix: GoalMixConfig, config: QiskitHybridConfig) -> ReducedUniverse:
    config.validate()
    if config.selection_objective == 'classical_target_recovery':
        table = build_active_screening_table(context=context, classical_reference=classical_reference, preferences=preferences)
    else:
        table = build_marginal_utility_screening_table(context=context, preferences=preferences, scales=scales, mix=mix, config=config)
    selected: list[str] = []

    def add(ticker: str) -> None:
        if ticker not in selected and len(selected) < config.max_qubits:
            selected.append(ticker)
    buys = table.loc[table['selection_trade'] > EPS].sort_values(['active_selection_score', 'absolute_selection_trade'], ascending=[False, False])
    sells = table.loc[table['selection_trade'] < -EPS].sort_values(['active_selection_score', 'absolute_selection_trade'], ascending=[False, False])
    side_slots = max(1, config.max_qubits // 3)
    for ticker in buys.head(side_slots).index:
        add(str(ticker))
    for ticker in sells.head(side_slots).index:
        add(str(ticker))
    for ticker in table.sort_values(['active_selection_score', 'absolute_selection_trade'], ascending=[False, False]).index:
        add(str(ticker))
    if len(selected) != config.max_qubits:
        raise RuntimeError('Could not construct the requested active qubit universe.')
    full_tickers = list(context.portfolio_data.tickers)
    indices = np.asarray([full_tickers.index(ticker) for ticker in selected], dtype=int)
    selected_table = table.loc[selected].copy()
    selected_table['qubit_index'] = np.arange(len(selected))
    return ReducedUniverse(tickers=selected, full_indices=indices, screening_table=selected_table, required_class_counts={}, minimum_feasible_cardinality=2)

def build_active_selection_model(*, context: Step5Context, classical_reference: dict[str, Any], reduced: ReducedUniverse, cardinality: int, config: QiskitHybridConfig) -> BinarySelectionModel:
    idx = reduced.full_indices
    k = int(cardinality)
    table = reduced.screening_table
    selection_trade = table['selection_trade'].to_numpy(dtype=float)
    trade_scale = max(float(np.max(np.abs(selection_trade))), 1e-08)
    normalized_trade = selection_trade / trade_scale
    trade_materiality = np.abs(normalized_trade)
    data = context.portfolio_data
    covariance = np.asarray(data.covariance, dtype=float)[np.ix_(idx, idx)]
    impact = np.asarray(data.impact_matrix, dtype=float)[np.ix_(idx, idx)]
    linear_cost = np.asarray(data.linear_cost, dtype=float)[idx]
    covariance_scale = max(float(np.max(np.abs(covariance))), 1e-12)
    impact_scale = max(float(np.max(np.abs(impact))), 1e-12)
    covariance_normalized = covariance / covariance_scale
    impact_normalized = impact / impact_scale
    impact_diagonal = np.diag(impact_normalized)
    cost_burden = unit_interval(linear_cost + impact_diagonal, True)
    active_score = table['active_selection_score'].to_numpy(dtype=float)
    score_scale = max(float(active_score.max()), 1e-12)
    normalized_score = active_score / score_scale
    linear = -normalized_score ** 2 + 0.05 * cost_burden * normalized_score
    Q = config.active_balance_strength * np.outer(normalized_trade, normalized_trade)
    trade_outer = np.outer(normalized_trade, normalized_trade)
    Q += 0.04 * trade_outer * covariance_normalized
    Q += 0.02 * trade_outer * impact_normalized
    Q = 0.5 * (Q + Q.T)
    labels = np.asarray(data.asset_classes, dtype=object)[idx]
    class_targets = largest_remainder_class_targets(labels=labels, required_counts={}, cardinality=k)
    return BinarySelectionModel(tickers=list(reduced.tickers), Q=Q, linear=linear, constant=0.0, cardinality=k, target_class_counts=class_targets)

def repair_active_selection(*, selected_indices: Any, reduced: ReducedUniverse, cardinality: int) -> np.ndarray:
    selected = set((int(index) for index in selected_indices))
    scores = reduced.screening_table['active_selection_score'].to_numpy(dtype=float)
    trades = reduced.screening_table['selection_trade'].to_numpy(dtype=float)
    ranked = list(np.argsort(scores)[::-1])
    for index in ranked:
        if len(selected) >= cardinality:
            break
        selected.add(int(index))
    while len(selected) > cardinality:
        outgoing = min(selected, key=lambda index: scores[index])
        selected.remove(outgoing)

    def ensure_sign(sign: int) -> None:
        if sign > 0:
            already = any((trades[index] > EPS for index in selected))
            options = [index for index in range(reduced.n_qubits) if index not in selected and trades[index] > EPS]
        else:
            already = any((trades[index] < -EPS for index in selected))
            options = [index for index in range(reduced.n_qubits) if index not in selected and trades[index] < -EPS]
        if already or not options:
            return
        incoming = max(options, key=lambda index: scores[index])
        removable = [index for index in selected if (trades[index] <= EPS if sign > 0 else trades[index] >= -EPS)]
        if not removable:
            removable = list(selected)
        outgoing = min(removable, key=lambda index: scores[index])
        selected.remove(outgoing)
        selected.add(incoming)
    ensure_sign(+1)
    ensure_sign(-1)
    bits = np.zeros(reduced.n_qubits, dtype=int)
    bits[list(selected)] = 1
    return bits

def clone_constraints_for_active_set(*, constraints: Any, current_weights: Any, active_full_mask: np.ndarray) -> Any:
    current = as_numpy(current_weights)
    lower = np.asarray(constraints.asset_lower, dtype=float).copy()
    upper = np.asarray(constraints.asset_upper, dtype=float).copy()
    inactive = ~active_full_mask
    if np.any(current[inactive] < lower[inactive] - 1e-10) or np.any(current[inactive] > upper[inactive] + 1e-10):
        raise ValueError('An inactive current weight lies outside the final asset bounds.')
    lower[inactive] = current[inactive]
    upper[inactive] = current[inactive]
    return replace(constraints, asset_lower=lower, asset_upper=upper)

def project_active_warm_start(*, source_weights: Any, current_weights: Any, active_full_mask: np.ndarray, lower: np.ndarray, upper: np.ndarray) -> np.ndarray:
    current = as_numpy(current_weights)
    target = as_numpy(source_weights).copy()
    inactive = ~active_full_mask
    target[inactive] = current[inactive]
    active_indices = np.flatnonzero(active_full_mask)
    target[active_indices] = np.clip(target[active_indices], lower[active_indices], upper[active_indices])
    required_active_sum = float(current[active_indices].sum())
    for _ in range(300):
        active_sum = float(target[active_indices].sum())
        difference = required_active_sum - active_sum
        if abs(difference) <= 1e-11:
            break
        if difference > 0.0:
            slack = np.maximum(upper[active_indices] - target[active_indices], 0.0)
        else:
            slack = np.maximum(target[active_indices] - lower[active_indices], 0.0)
        total_slack = float(slack.sum())
        if total_slack <= EPS:
            target = current.copy()
            break
        target[active_indices] += difference * slack / total_slack
        target[active_indices] = np.clip(target[active_indices], lower[active_indices], upper[active_indices])
    target[inactive] = current[inactive]
    if abs(float(target.sum()) - 1.0) > 1e-08:
        target = current.copy()
    return target

def refine_active_subset(*, context: Step5Context, classical_reference: dict[str, Any], reduced: ReducedUniverse, bits: Any, preferences: GoalPreferences, scales: GoalScales, mix: GoalMixConfig, config: QiskitHybridConfig, label: str) -> dict[str, Any]:
    bit_array = np.asarray(bits, dtype=int)
    active_full_mask = np.zeros(len(context.portfolio_data.tickers), dtype=bool)
    active_full_mask[reduced.full_indices[bit_array == 1]] = True
    active_constraints = clone_constraints_for_active_set(constraints=context.constraints, current_weights=context.current_weights, active_full_mask=active_full_mask)
    active_context = replace(context, constraints=active_constraints)
    warm_source = classical_reference['result'].weights if config.selection_objective == 'classical_target_recovery' else context.current_weights
    warm_start = project_active_warm_start(source_weights=warm_source, current_weights=context.current_weights, active_full_mask=active_full_mask, lower=np.asarray(active_constraints.asset_lower, dtype=float), upper=np.asarray(active_constraints.asset_upper, dtype=float))
    solved = solve_goal_profile(context=active_context, profile_name=label, preferences=preferences, scales=scales, warm_start=warm_start, mix=mix)
    exact = normalized_step5_objective(context=context, weights=solved['result'].weights, preferences=preferences, scales=scales, mix=mix)
    solved['hybrid_normalized_objective'] = exact['normalized_objective']
    solved['selected_tickers'] = [context.portfolio_data.tickers[index] for index in np.flatnonzero(active_full_mask)]
    solved['selected_full_mask'] = active_full_mask
    solved['selection_mode'] = 'active_rebalance'
    solved['selection_objective'] = config.selection_objective
    final_weights = as_numpy(solved['result'].weights)
    current_weights = as_numpy(context.current_weights)
    executed_mask = np.abs(final_weights - current_weights) > config.executed_trade_threshold
    solved['executed_trade_tickers'] = [context.portfolio_data.tickers[index] for index in np.flatnonzero(executed_mask)]
    solved['executed_trade_count'] = int(executed_mask.sum())
    solved['nonzero_holding_count'] = int(np.count_nonzero(final_weights > 1e-06))
    return solved

def normalized_step5_objective(*, context: Step5Context, weights: Any, preferences: GoalPreferences, scales: GoalScales, mix: GoalMixConfig) -> dict[str, float]:
    components = exact_goal_components(context.portfolio_data, weights, context.current_weights, context.scenarios)
    coefficients = derive_step5_coefficients(preferences, scales, mix)
    objective = -coefficients['growth'] * components['growth'] - coefficients['income'] * components['income'] + coefficients['variance'] * components['variance'] + coefficients['scenario'] * components['scenario_hinge'] + coefficients['linear_cost'] * components['linear_cost'] + coefficients['impact'] * components['impact_cost'] + coefficients['turnover'] * components['gross_turnover'] + coefficients['concentration'] * components['concentration']
    return {'normalized_objective': float(objective), **components}

def refine_subset(*, context: Step5Context, classical_reference: dict[str, Any], reduced: ReducedUniverse, bits: Any, preferences: GoalPreferences, scales: GoalScales, mix: GoalMixConfig, config: QiskitHybridConfig, label: str) -> dict[str, Any]:
    bit_array = np.asarray(bits, dtype=int)
    selected_full_mask = np.zeros(len(context.portfolio_data.tickers), dtype=bool)
    selected_full_mask[reduced.full_indices[bit_array == 1]] = True
    subset_constraints = clone_constraints_for_subset(constraints=context.constraints, selected_full_mask=selected_full_mask, selected_minimum_weight=config.selected_minimum_weight)
    subset_context = replace(context, constraints=subset_constraints)
    warm_start = project_warm_start(source_weights=classical_reference['result'].weights, selected_full_mask=selected_full_mask, lower=np.asarray(subset_constraints.asset_lower, dtype=float), upper=np.asarray(subset_constraints.asset_upper, dtype=float))
    solved = solve_goal_profile(context=subset_context, profile_name=label, preferences=preferences, scales=scales, warm_start=warm_start, mix=mix)
    exact = normalized_step5_objective(context=context, weights=solved['result'].weights, preferences=preferences, scales=scales, mix=mix)
    solved['hybrid_normalized_objective'] = exact['normalized_objective']
    solved['selected_tickers'] = [context.portfolio_data.tickers[index] for index in np.flatnonzero(selected_full_mask)]
    solved['selected_full_mask'] = selected_full_mask
    return solved

def run_hybrid_qiskit_pipeline(*, context: Step5Context, preferences: GoalPreferences, scales: GoalScales, mix: GoalMixConfig, classical_reference: dict[str, Any], config: QiskitHybridConfig) -> dict[str, Any]:
    config.validate()
    if config.selection_mode == 'active_rebalance':
        reduced = reduce_active_universe(context=context, classical_reference=classical_reference, preferences=preferences, scales=scales, mix=mix, config=config)
        cardinality = infer_active_cardinality(reduced=reduced, config=config) if config.cardinality is None else int(config.cardinality)
        if cardinality < 2:
            raise ValueError('Active-rebalance cardinality must be at least two.')
        if cardinality > reduced.n_qubits:
            raise ValueError('cardinality cannot exceed n_qubits.')
        model = build_active_selection_model(context=context, classical_reference=classical_reference, reduced=reduced, cardinality=cardinality, config=config)

        def repair(bits: Any) -> np.ndarray:
            return repair_active_selection(selected_indices=np.flatnonzero(np.asarray(bits, dtype=int)), reduced=reduced, cardinality=cardinality)

        def refine(bits: Any, label: str) -> dict[str, Any]:
            return refine_active_subset(context=context, classical_reference=classical_reference, reduced=reduced, bits=bits, preferences=preferences, scales=scales, mix=mix, config=config, label=label)
    else:
        reduced = reduce_universe(context=context, preferences=preferences, config=config)
        cardinality = reduced.minimum_feasible_cardinality if config.cardinality is None else int(config.cardinality)
        if cardinality < reduced.minimum_feasible_cardinality:
            raise ValueError(f'cardinality={cardinality} is below the policy-implied minimum {reduced.minimum_feasible_cardinality}.')
        if cardinality > reduced.n_qubits:
            raise ValueError('cardinality cannot exceed n_qubits.')
        model = build_binary_selection_model(context=context, reduced=reduced, preferences=preferences, scales=scales, mix=mix, cardinality=cardinality, config=config)

        def repair(bits: Any) -> np.ndarray:
            return repair_selection(selected_indices=np.flatnonzero(np.asarray(bits, dtype=int)), reduced=reduced, context=context, cardinality=cardinality)

        def refine(bits: Any, label: str) -> dict[str, Any]:
            return refine_subset(context=context, classical_reference=classical_reference, reduced=reduced, bits=bits, preferences=preferences, scales=scales, mix=mix, config=config, label=label)
    qp = build_qiskit_quadratic_program(model)
    exact_enumeration = enumerate_fixed_cardinality(model, config.exact_enumeration_limit)
    qiskit_run = run_qiskit_qaoa(quadratic_program=qp, model=model, config=config)
    if not exact_enumeration.empty:
        exact_string = str(exact_enumeration.iloc[0]['bitstring'])
        exact_bits = np.fromiter((int(value) for value in exact_string), dtype=int)
        qiskit_run['exact_bits'] = exact_bits
        qiskit_run['exact_energy'] = float(exact_enumeration.iloc[0]['energy'])
    refinement_records: list[dict[str, Any]] = []
    successful_qaoa: list[dict[str, Any]] = []
    successful_fallback: list[dict[str, Any]] = []
    seen: set[str] = set()

    def attempt_candidate(*, bits: np.ndarray, sample_rank: int, probability: float, source: str) -> None:
        repaired = repair(bits)
        bitstring = ''.join((str(int(value)) for value in repaired))
        if bitstring in seen:
            return
        seen.add(bitstring)
        try:
            solved = refine(repaired, f'Qiskit QAOA sample {sample_rank}' if source == 'qaoa' else f'Exact-enumeration feasibility fallback {sample_rank}')
            solved['selection_source'] = source
            target = successful_qaoa if source == 'qaoa' else successful_fallback
            target.append(solved)
            refinement_records.append({'sample_rank': int(sample_rank), 'selection_source': source, 'bitstring': bitstring, 'probability': float(probability), 'qubo_energy': model.energy(repaired), 'success': True, 'normalized_continuous_objective': solved['hybrid_normalized_objective'], 'expected_total_return': solved['metrics']['expected_total_return'], 'volatility': solved['metrics']['volatility'], 'worst_scenario_loss': solved['metrics']['worst_scenario_loss'], 'gross_turnover': solved['metrics']['gross_turnover'], 'total_trading_cost': solved['metrics']['total_trading_cost'], 'selected_tickers': ', '.join(solved['selected_tickers']), 'message': solved['result'].message})
        except Exception as error:
            refinement_records.append({'sample_rank': int(sample_rank), 'selection_source': source, 'bitstring': bitstring, 'probability': float(probability), 'qubo_energy': model.energy(repaired), 'success': False, 'message': f'{type(error).__name__}: {error}'})
    for sample_rank, sample in qiskit_run['samples'].iterrows():
        if len(successful_qaoa) >= config.maximum_subsets_to_refine:
            break
        bits = np.zeros(reduced.n_qubits, dtype=int)
        bits[list(sample['selected_indices'])] = 1
        attempt_candidate(bits=bits, sample_rank=int(sample_rank), probability=float(sample.get('conditional_probability', sample.get('raw_solver_probability', 0.0))), source='qaoa')
    if not exact_enumeration.empty:
        exact_limit = min(len(exact_enumeration), config.exact_candidates_to_refine)
        for exact_rank, row in exact_enumeration.head(exact_limit).iterrows():
            bits = np.fromiter((int(value) for value in str(row['bitstring'])), dtype=int)
            attempt_candidate(bits=bits, sample_rank=int(exact_rank), probability=0.0, source='exact_active_set_benchmark')
    candidate_pool = successful_qaoa if successful_qaoa else successful_fallback
    if not candidate_pool:
        failure_table = pd.DataFrame(refinement_records)
        messages = failure_table.get('message', pd.Series(dtype=str)).astype(str).head(5).tolist()
        raise RuntimeError('No candidate subset produced a feasible continuous portfolio. First failures: ' + ' | '.join(messages))
    hybrid = min(candidate_pool, key=lambda result: result['hybrid_normalized_objective'])
    source = hybrid.get('selection_source', 'qaoa')
    if config.selection_mode == 'active_rebalance':
        profile = 'Hybrid Qiskit QAOA Active Rebalance' if source == 'qaoa' else 'Hybrid Active Rebalance (Exact Active-Set Benchmark)'
    else:
        profile = 'Hybrid Qiskit QAOA' if source == 'qaoa' else 'Hybrid Support Selection (Exact Feasibility Fallback)'
    hybrid['profile'] = profile
    hybrid['result'].stage = profile
    hybrid['selection_mode'] = config.selection_mode
    exact_refinement = None
    exact_bits = qiskit_run.get('exact_bits')
    try:
        if exact_bits is None:
            raise ValueError('No exact benchmark bitstring was available.')
        exact_refinement = refine(repair(exact_bits), 'Exact reduced-QUBO benchmark')
    except Exception:
        exact_refinement = None
    exact_active_profile = None
    if exact_refinement is not None:
        exact_refinement['selection_source'] = 'exact_active_set_benchmark'
        exact_refinement['profile'] = 'Exact Active-Set Benchmark'
        exact_refinement['result'].stage = 'Exact Active-Set Benchmark'
        exact_refinement['selection_mode'] = config.selection_mode
        exact_refinement['selection_objective'] = config.selection_objective
        exact_active_profile = exact_refinement
    best_additional_exact_profile = min(successful_fallback, key=lambda result: result['hybrid_normalized_objective']) if successful_fallback else None
    best_available_profile = min([result for result in (hybrid, exact_active_profile, best_additional_exact_profile) if result is not None], key=lambda result: result['hybrid_normalized_objective'])
    return {'config': config, 'selection_mode': config.selection_mode, 'selection_objective': config.selection_objective, 'reduced_universe': reduced, 'selection_model': model, 'quadratic_program': qp, 'exact_enumeration': exact_enumeration, 'qiskit_run': qiskit_run, 'refinement_table': pd.DataFrame(refinement_records), 'hybrid_profile_result': hybrid, 'exact_reduced_qubo_refinement': exact_refinement, 'exact_active_profile_result': exact_active_profile, 'best_additional_exact_profile_result': best_additional_exact_profile, 'best_available_profile_result': best_available_profile, 'cardinality': cardinality, 'qaoa_feasible_candidate_found': bool(successful_qaoa), 'selection_source': source}

def greedy_fixed_cardinality_bits(model: BinarySelectionModel) -> np.ndarray:
    selected: list[int] = []
    n = len(model.tickers)
    while len(selected) < model.cardinality:
        best_index = None
        best_energy = np.inf
        for candidate in range(n):
            if candidate in selected:
                continue
            bits = np.zeros(n, dtype=int)
            bits[selected + [candidate]] = 1
            energy = model.energy(bits)
            if energy < best_energy:
                best_energy = energy
                best_index = candidate
        if best_index is None:
            raise RuntimeError('Greedy subset construction failed.')
        selected.append(best_index)
    bits = np.zeros(n, dtype=int)
    bits[selected] = 1
    return bits

def local_search_fixed_cardinality_bits(model: BinarySelectionModel, initial_bits: Any | None=None) -> np.ndarray:
    bits = greedy_fixed_cardinality_bits(model) if initial_bits is None else np.asarray(initial_bits, dtype=int).copy()
    if int(bits.sum()) != model.cardinality:
        raise ValueError('initial_bits has the wrong cardinality.')
    improved = True
    while improved:
        improved = False
        current_energy = model.energy(bits)
        selected = np.flatnonzero(bits == 1)
        unselected = np.flatnonzero(bits == 0)
        best_swap = None
        best_energy = current_energy
        for outgoing in selected:
            for incoming in unselected:
                trial = bits.copy()
                trial[outgoing] = 0
                trial[incoming] = 1
                energy = model.energy(trial)
                if energy < best_energy - 1e-12:
                    best_energy = energy
                    best_swap = (outgoing, incoming)
        if best_swap is not None:
            bits[best_swap[0]] = 0
            bits[best_swap[1]] = 1
            improved = True
    return bits

def random_fixed_cardinality_candidates(*, n_qubits: int, cardinality: int, number_of_samples: int, seed: int) -> list[np.ndarray]:
    rng = np.random.default_rng(seed)
    total = math.comb(n_qubits, cardinality)
    target = min(int(number_of_samples), total)
    seen: set[tuple[int, ...]] = set()
    candidates: list[np.ndarray] = []
    while len(candidates) < target:
        chosen = tuple(sorted((int(value) for value in rng.choice(n_qubits, size=cardinality, replace=False))))
        if chosen in seen:
            continue
        seen.add(chosen)
        bits = np.zeros(n_qubits, dtype=int)
        bits[list(chosen)] = 1
        candidates.append(bits)
    return candidates

def evaluate_classical_subset_baselines(*, context: Step5Context, classical_reference: dict[str, Any], reduced: ReducedUniverse, model: BinarySelectionModel, preferences: GoalPreferences, scales: GoalScales, mix: GoalMixConfig, config: QiskitHybridConfig, random_budget: int=20, seed: int=12345) -> tuple[pd.DataFrame, dict[str, dict[str, Any]]]:
    candidate_map: dict[str, np.ndarray] = {'greedy': greedy_fixed_cardinality_bits(model)}
    candidate_map['local_search'] = local_search_fixed_cardinality_bits(model, candidate_map['greedy'])
    for index, bits in enumerate(random_fixed_cardinality_candidates(n_qubits=len(model.tickers), cardinality=model.cardinality, number_of_samples=random_budget, seed=seed)):
        candidate_map[f'random_{index:03d}'] = bits
    records: list[dict[str, Any]] = []
    solved_profiles: dict[str, dict[str, Any]] = {}
    for name, bits in candidate_map.items():
        try:
            solved = refine_active_subset(context=context, classical_reference=classical_reference, reduced=reduced, bits=bits, preferences=preferences, scales=scales, mix=mix, config=config, label=f'Classical subset baseline: {name}')
            solved_profiles[name] = solved
            records.append({'selector': name, 'success': True, 'bitstring': ''.join((str(int(x)) for x in bits)), 'qubo_energy': model.energy(bits), 'normalized_objective': solved['hybrid_normalized_objective'], 'expected_total_return': solved['metrics']['expected_total_return'], 'volatility': solved['metrics']['volatility'], 'worst_scenario_loss': solved['metrics']['worst_scenario_loss'], 'gross_turnover': solved['metrics']['gross_turnover'], 'total_trading_cost': solved['metrics']['total_trading_cost'], 'selected_tickers': ', '.join(solved['selected_tickers'])})
        except Exception as error:
            records.append({'selector': name, 'success': False, 'bitstring': ''.join((str(int(x)) for x in bits)), 'qubo_energy': model.energy(bits), 'message': f'{type(error).__name__}: {error}'})
    return (pd.DataFrame(records), solved_profiles)


Writing step_05q_hybrid_qaoa_final.py


## Verify the installed Qiskit API versions

In [6]:
import qiskit
import qiskit_algorithms
import qiskit_optimization
import qiskit_aer
print('Qiskit:', qiskit.__version__)
print('Qiskit Algorithms:', qiskit_algorithms.__version__)
print('Qiskit Optimization:', qiskit_optimization.__version__)
print('Qiskit Aer:', qiskit_aer.__version__)
from qiskit_aer.primitives import SamplerV2 as AerSamplerV2
from qiskit_algorithms import QAOA
from qiskit_optimization.algorithms import MinimumEigenOptimizer
print('Aer SamplerV2 and QAOA imports passed.')
print('NumPyMinimumEigensolver is intentionally not used in this notebook.')


Qiskit: 2.5.1
Qiskit Algorithms: 0.4.0
Qiskit Optimization: 0.7.0
Qiskit Aer: 0.17.2
Aer SamplerV2 and QAOA imports passed.
NumPyMinimumEigensolver is intentionally not used in this notebook.


## Configure browser-only checkpointing

In [7]:
from pathlib import Path
import importlib
RUN_MODE = 'fresh'
AUTO_DOWNLOAD_BUNDLES = True
CHECKPOINT_ROOT = Path('/content/quantum_portfolio_checkpoints')
PRE_QAOA_CHECKPOINT = CHECKPOINT_ROOT / 'pre_qaoa_checkpoint.pkl.gz'
QAOA_RUN_DIR = CHECKPOINT_ROOT / 'qaoa_seed_runs'
QAOA_PROGRESS_DIR = CHECKPOINT_ROOT / 'qaoa_progress'
for directory in (CHECKPOINT_ROOT, QAOA_RUN_DIR, QAOA_PROGRESS_DIR):
    directory.mkdir(parents=True, exist_ok=True)
if RUN_MODE not in {'fresh', 'resume'}:
    raise ValueError("RUN_MODE must be 'fresh' or 'resume'.")
print('Run mode:', RUN_MODE)
print('Google Drive is not mounted.')


Run mode: fresh
Google Drive is not mounted.


## Upload and restore the latest resume bundle

In [8]:
import cloudpickle
import gzip
import zipfile
if RUN_MODE == 'resume':
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(zip_names) != 1:
        raise ValueError('Upload exactly one resume-bundle ZIP.')
    uploaded_zip = Path('/content') / zip_names[0]
    with zipfile.ZipFile(uploaded_zip, 'r') as archive:
        archive.extractall('/content')
    CHECKPOINT_ROOT = Path('/content/quantum_portfolio_checkpoints')
    PRE_QAOA_CHECKPOINT = CHECKPOINT_ROOT / 'pre_qaoa_checkpoint.pkl.gz'
    QAOA_RUN_DIR = CHECKPOINT_ROOT / 'qaoa_seed_runs'
    QAOA_PROGRESS_DIR = CHECKPOINT_ROOT / 'qaoa_progress'
    if not PRE_QAOA_CHECKPOINT.exists():
        legacy_checkpoint = CHECKPOINT_ROOT / 'step3_step4_step5_pre_qaoa.pkl.gz'
        if legacy_checkpoint.exists():
            PRE_QAOA_CHECKPOINT = legacy_checkpoint
            print('Using compatible legacy classical checkpoint:', PRE_QAOA_CHECKPOINT)
        else:
            raise FileNotFoundError('The uploaded ZIP has no compatible pre-QAOA checkpoint.')
    import step_03_data_pipeline_final as step3
    import step_04_classical_baseline_final as step4
    import step_05_tunable_goals_final as step5
    with gzip.open(PRE_QAOA_CHECKPOINT, 'rb') as file:
        payload = cloudpickle.load(file)
    globals().update(payload)
    FAST_MODE = globals().get('FAST_MODE', True)
    RISK_POLICY_MODE = globals().get('RISK_POLICY_MODE', 'soft_warning')
    RUN_INSTITUTIONAL_SENSITIVITY = globals().get('RUN_INSTITUTIONAL_SENSITIVITY', False)
    RUN_FORWARD_SIMULATION = globals().get('RUN_FORWARD_SIMULATION', True)
    step3 = importlib.reload(step3)
    step4 = importlib.reload(step4)
    step5 = importlib.reload(step5)
    step5.patch_step4_exact_hinge_reporting(step4)
    import importlib.util
    import inspect
    import sys
    MODULE_NAME = 'step_05q_hybrid_qaoa_final'
    MODULE_PATH = Path('/content') / f'{MODULE_NAME}.py'
    sys.modules.pop(MODULE_NAME, None)
    importlib.invalidate_caches()
    _spec = importlib.util.spec_from_file_location(MODULE_NAME, MODULE_PATH)
    if _spec is None or _spec.loader is None:
        raise ImportError('Could not load the release hybrid module.')
    hybrid = importlib.util.module_from_spec(_spec)
    sys.modules[MODULE_NAME] = hybrid
    _spec.loader.exec_module(hybrid)
    _pipeline_source = inspect.getsource(hybrid.run_hybrid_qiskit_pipeline)
    assert 'active_rebalance' in _pipeline_source
    assert 'exact_active_set_benchmark' in _pipeline_source
    assert 'best_available_profile_result' in _pipeline_source
    STEP5_CONTEXT = step5.Step5Context(step4=step4, portfolio_data=portfolio_data, current_weights=current_weights, stages=stages, scenarios=scenarios, constraints=constraints, trading_config=trading_config, daily_returns=STEP5_DAILY_RETURNS)
    print('Resume bundle restored.')
    print('Completed seed files:', len(list(QAOA_RUN_DIR.glob('qaoa_independent_seed_*.pkl.gz'))))
else:
    print('Fresh mode selected.')


Fresh mode selected.


# Step 5Q-A — Build the common goal objective and classical reference

In [17]:
if RUN_MODE == 'fresh':
    import gc
    import os
    import psutil
    import matplotlib.pyplot as plt
    plt.close('all')
    for _name in ['downloaded_prices', 'raw_prices', 'price_panel', 'returns_panel', 'synthetic_prices']:
        if _name in globals():
            del globals()[_name]
    gc.collect()
    _process = psutil.Process(os.getpid())
    print('RAM before Step 5Q:', f'{_process.memory_info().rss / 1024 ** 3:.2f} GB')
else:
    print('Skipped pre-QAOA RAM cleanup: restored from the uploaded bundle.')


RAM before Step 5Q: 0.32 GB


In [18]:
if RUN_MODE == 'fresh':
    import importlib
    import pandas as pd
    import numpy as np
    import step_05_tunable_goals_final as step5
    step5 = importlib.reload(step5)
    step5.patch_step4_exact_hinge_reporting(step4)
    daily_return_candidates = [SOURCE_DIR / f'{PREFIX}_daily_returns_raw.csv', SOURCE_DIR / f'{PREFIX}_daily_returns.csv']
    daily_return_file = next((path for path in daily_return_candidates if path.exists()), None)
    if daily_return_file is None:
        raise FileNotFoundError('No Step 3 daily-return file was found. Checked: ' + ', '.join((str(path) for path in daily_return_candidates)))
    STEP5_DAILY_RETURNS = pd.read_csv(daily_return_file, index_col=0, parse_dates=True)
    STEP5_DAILY_RETURNS = STEP5_DAILY_RETURNS.loc[:, portfolio_data.tickers].dropna(how='all')
    STEP5_CONTEXT = step5.Step5Context(step4=step4, portfolio_data=portfolio_data, current_weights=current_weights, stages=stages, scenarios=scenarios, constraints=constraints, trading_config=trading_config, daily_returns=STEP5_DAILY_RETURNS)
    GROWTH_SCORE = 55
    INCOME_SCORE = 55
    DRAWDOWN_SCORE = 70
    COST_SCORE = 60
    HYBRID_PREFERENCES = step5.GoalPreferences(growth=GROWTH_SCORE, income=INCOME_SCORE, drawdown_control=DRAWDOWN_SCORE, cost_sensitivity=COST_SCORE)
    print('Preference shares:')
    display(pd.Series(HYBRID_PREFERENCES.shares, name='relative_share').to_frame().style.format('{:.1%}'))
    print('Solving fully constrained anchors...')
    FULLY_CONSTRAINED_ANCHORS = step5.solve_fully_constrained_anchors(STEP5_CONTEXT)
    GOAL_SCALES, GOAL_SCALE_CALIBRATION = step5.calibrate_goal_scales_from_feasible_set(STEP5_CONTEXT, FULLY_CONSTRAINED_ANCHORS)
    print('Solving unrestricted classical reference under the same preferences...')
    CLASSICAL_REFERENCE = step5.solve_goal_profile(context=STEP5_CONTEXT, profile_name='Unrestricted classical reference', preferences=HYBRID_PREFERENCES, scales=GOAL_SCALES, mix=step5.GOAL_MIX)
    display(GOAL_SCALE_CALIBRATION)
    display(pd.Series(CLASSICAL_REFERENCE['metrics'], name='value').to_frame())
    display(CLASSICAL_REFERENCE['result'].weights.sort_values(ascending=False).head(20).to_frame('weight').style.format('{:.2%}'))
else:
    print('Skipped Step 5 goal preparation: restored from the uploaded bundle.')


Preference shares:


,relative_share
growth,22.9%
income,22.9%
drawdown,29.2%
cost,25.0%


Solving fully constrained anchors...
Solving unrestricted classical reference under the same preferences...


,growth,income,expected_total_return,variance,volatility,scenario_hinge,worst_scenario_loss,linear_cost,impact_cost,total_trading_cost,gross_turnover,one_way_turnover,concentration,effective_holdings,maximum_asset_weight
anchor,,,,,,,,,,,,,,,
current_portfolio,0.031423,0.024028,0.055451,0.007784,0.088228,1.181250e-03,0.138270,0.000000,0.000000,0.000000,0.0,0.00,0.024911,40.142820,0.050000
step4_scenario_aware,0.026507,0.027273,0.053780,0.003523,0.059354,9.987585e-07,0.103633,0.000100,0.000095,0.000195,0.5,0.25,0.048303,20.702440,0.134292
maximum_feasible_growth,0.042345,0.021028,0.063373,0.012072,0.109874,4.332658e-03,0.157435,0.000138,0.000336,0.000474,0.5,0.25,0.043054,23.226376,0.097813
maximum_feasible_income,0.025571,0.031686,0.057257,0.007531,0.086779,4.667434e-03,0.144091,0.000117,0.000202,0.000318,0.5,0.25,0.042997,23.257228,0.096648
minimum_feasible_variance,0.023592,0.024183,0.047775,0.002549,0.050488,1.296599e-04,0.088584,0.000089,0.000076,0.000165,0.5,0.25,0.050872,19.657318,0.134292


,value
growth,0.028985
income,0.025911
expected_total_return,0.054896
variance,0.006180
volatility,0.078614
scenario_hinge,0.000183
worst_scenario_loss,0.127113
linear_cost,0.000013
impact_cost,0.000007
total_trading_cost,0.000020


,weight
SGOV,7.97%
TIP,5.00%
BWX,3.98%
AGG,3.68%
BNDX,3.68%
BND,3.66%
USMV,3.45%
TLT,3.41%
SHY,3.40%
IEF,3.19%


## Strict scenario-warning comparison

In [19]:
from dataclasses import replace as dataclass_replace
STRICT_WARNING_REFERENCE = None
STRICT_WARNING_CONTEXT = None
STRICT_WARNING_ERROR = None
try:
    strict_scenarios = dataclass_replace(scenarios, hard_loss_limits=np.asarray(scenarios.warning_thresholds, dtype=float).copy())
    STRICT_WARNING_CONTEXT = dataclass_replace(STEP5_CONTEXT, scenarios=strict_scenarios)
    STRICT_WARNING_REFERENCE = step5.solve_goal_profile(context=STRICT_WARNING_CONTEXT, profile_name='Classical strict-warning reference', preferences=HYBRID_PREFERENCES, scales=GOAL_SCALES, mix=step5.GOAL_MIX)
    strict_losses = strict_scenarios.loss_matrix @ STRICT_WARNING_REFERENCE['result'].weights.to_numpy(dtype=float)
    assert np.all(strict_losses <= strict_scenarios.warning_thresholds + 5e-06)
    print('Strict-warning portfolio solved successfully.')
    display(pd.Series(STRICT_WARNING_REFERENCE['metrics'], name='value').to_frame())
except Exception as error:
    STRICT_WARNING_ERROR = f'{type(error).__name__}: {error}'
    print('Strict-warning profile unavailable:', STRICT_WARNING_ERROR)


Strict-warning portfolio solved successfully.


,value
growth,2.819180e-02
income,2.665070e-02
expected_total_return,5.484250e-02
variance,5.633895e-03
volatility,7.505928e-02
scenario_hinge,2.265134e-30
worst_scenario_loss,1.216857e-01
linear_cost,2.885470e-05
impact_cost,2.668239e-05
total_trading_cost,5.553709e-05


## Select the primary risk policy

In [20]:
if RISK_POLICY_MODE not in {'soft_warning', 'strict_warning'}:
    raise ValueError("RISK_POLICY_MODE must be 'soft_warning' or 'strict_warning'.")
if RISK_POLICY_MODE == 'strict_warning':
    if STRICT_WARNING_REFERENCE is None:
        raise RuntimeError('Strict-warning policy was selected, but the strict-warning classical problem was not feasible.')
    PRIMARY_CONTEXT = STRICT_WARNING_CONTEXT
    PRIMARY_CLASSICAL_REFERENCE = STRICT_WARNING_REFERENCE
    PRIMARY_SCENARIOS = STRICT_WARNING_CONTEXT.scenarios
else:
    PRIMARY_CONTEXT = STEP5_CONTEXT
    PRIMARY_CLASSICAL_REFERENCE = CLASSICAL_REFERENCE
    PRIMARY_SCENARIOS = scenarios
print('Primary risk policy:', RISK_POLICY_MODE)
print('Primary classical reference:', PRIMARY_CLASSICAL_REFERENCE['profile'])


Primary risk policy: soft_warning
Primary classical reference: Unrestricted classical reference


## Browser resume-bundle helper

In [21]:
from datetime import datetime, timezone
import shutil

def create_resume_bundle(label: str, download: bool=True) -> Path:
    timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    safe_label = label.replace(' ', '_').replace('/', '_')
    archive_base = Path('/content') / f'quantum_portfolio_resume_bundle_{safe_label}_{timestamp}'
    archive_path = Path(shutil.make_archive(str(archive_base), 'zip', root_dir='/content', base_dir=CHECKPOINT_ROOT.name))
    print('Created resume bundle:', archive_path)
    print('Size:', f'{archive_path.stat().st_size / 1024 ** 2:.2f} MB')
    if download:
        from google.colab import files
        files.download(str(archive_path))
    return archive_path


## Save the pre-QAOA checkpoint

In [22]:
import cloudpickle
import gzip
if RUN_MODE == 'fresh':
    payload = {'OUTPUT_ROOT': OUTPUT_ROOT, 'STEP3_OUTPUT': STEP3_OUTPUT, 'STEP4_OUTPUT': STEP4_OUTPUT, 'DATA_SOURCE': DATA_SOURCE, 'SOURCE_DIR': SOURCE_DIR, 'PREFIX': PREFIX, 'STEP4_COST_SCENARIO': STEP4_COST_SCENARIO, 'portfolio_data': portfolio_data, 'current_weights': current_weights, 'objective_weights': objective_weights, 'trading_config': trading_config, 'solver_config': solver_config, 'stages': stages, 'scenarios': scenarios, 'constraints': constraints, 'RESULT_DIR': RESULT_DIR, 'STEP5_DAILY_RETURNS': STEP5_DAILY_RETURNS, 'GROWTH_SCORE': GROWTH_SCORE, 'INCOME_SCORE': INCOME_SCORE, 'DRAWDOWN_SCORE': DRAWDOWN_SCORE, 'COST_SCORE': COST_SCORE, 'HYBRID_PREFERENCES': HYBRID_PREFERENCES, 'FULLY_CONSTRAINED_ANCHORS': FULLY_CONSTRAINED_ANCHORS, 'GOAL_SCALES': GOAL_SCALES, 'GOAL_SCALE_CALIBRATION': GOAL_SCALE_CALIBRATION, 'CLASSICAL_REFERENCE': CLASSICAL_REFERENCE, 'STRICT_WARNING_REFERENCE': STRICT_WARNING_REFERENCE, 'STRICT_WARNING_CONTEXT': STRICT_WARNING_CONTEXT, 'RISK_POLICY_MODE': RISK_POLICY_MODE, 'PRIMARY_CONTEXT': PRIMARY_CONTEXT, 'PRIMARY_CLASSICAL_REFERENCE': PRIMARY_CLASSICAL_REFERENCE, 'PRIMARY_SCENARIOS': PRIMARY_SCENARIOS, 'FAST_MODE': FAST_MODE, 'RUN_INSTITUTIONAL_SENSITIVITY': RUN_INSTITUTIONAL_SENSITIVITY, 'RUN_FORWARD_SIMULATION': RUN_FORWARD_SIMULATION}
    temporary = PRE_QAOA_CHECKPOINT.with_suffix('.tmp')
    with gzip.open(temporary, 'wb') as file:
        cloudpickle.dump(payload, file)
    temporary.replace(PRE_QAOA_CHECKPOINT)
    print('Saved local checkpoint:', PRE_QAOA_CHECKPOINT)
    print('Size:', f'{PRE_QAOA_CHECKPOINT.stat().st_size / 1024 ** 2:.2f} MB')
    if AUTO_DOWNLOAD_BUNDLES:
        LATEST_RESUME_BUNDLE = create_resume_bundle('pre_qaoa', download=True)
else:
    print('Using checkpoint restored from the uploaded bundle:', PRE_QAOA_CHECKPOINT)


Saved local checkpoint: /content/quantum_portfolio_checkpoints/portfolio_pipeline_pre_qaoa.pkl.gz
Size: 0.51 MB
Created resume bundle: /content/quantum_portfolio_resume_bundle_pre_qaoa_20260805T174111Z.zip
Size: 0.51 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Resumable QAOA reliability experiment

## Import the hybrid module before QAOA configuration

In [23]:
import importlib
import importlib.util
import inspect
import sys
from pathlib import Path
CONTENT_ROOT = Path('/content')
MODULE_NAME = 'step_05q_hybrid_qaoa_final'
MODULE_PATH = CONTENT_ROOT / f'{MODULE_NAME}.py'
if not MODULE_PATH.exists():
    raise FileNotFoundError(f'{MODULE_PATH} does not exist. Run the preceding release module writefile cell.')
sys.modules.pop(MODULE_NAME, None)
importlib.invalidate_caches()
_spec = importlib.util.spec_from_file_location(MODULE_NAME, MODULE_PATH)
if _spec is None or _spec.loader is None:
    raise ImportError('Could not create the release module import specification.')
hybrid = importlib.util.module_from_spec(_spec)
sys.modules[MODULE_NAME] = hybrid
_spec.loader.exec_module(hybrid)
required_objects = ['QiskitHybridConfig', 'run_hybrid_qiskit_pipeline', 'reduce_active_universe', 'build_active_selection_model', 'refine_active_subset']
missing = [name for name in required_objects if not hasattr(hybrid, name)]
if missing:
    raise ImportError('The loaded release module is missing: ' + ', '.join(missing))
_pipeline_source = inspect.getsource(hybrid.run_hybrid_qiskit_pipeline)
if 'No QAOA-measured subset produced a feasible continuous portfolio' in _pipeline_source:
    raise RuntimeError('An obsolete whole-support pipeline was loaded.')
for required_marker in ['active_rebalance', 'exact_active_set_benchmark', 'best_available_profile_result', 'refine_active_subset']:
    if required_marker not in _pipeline_source:
        raise RuntimeError(f'The release pipeline is missing marker: {required_marker}')
print('release hybrid module imported successfully.')
print('Module path:', Path(hybrid.__file__).resolve())
print('Module name:', hybrid.__name__)
print('Active-rebalance pipeline markers verified.')


Hybrid QAOA hybrid module imported successfully.
Module path: /content/step_05q_hybrid_qaoa_final.py
Module name: step_05q_hybrid_qaoa
Active-rebalance pipeline markers verified.


## Why the previous whole-support model failed

In [24]:
import numpy as np
import pandas as pd
_current = np.asarray(current_weights, dtype=float)
_limit = float(trading_config.turnover_limit_gross)
_required_mass = 1.0 - 0.5 * _limit
_sorted = np.sort(_current)[::-1]
_cumulative = np.cumsum(_sorted)
_positions = np.flatnonzero(_cumulative >= _required_mass - 1e-12)
_minimum_support = int(_positions[0] + 1)
_top_11_mass = float(_sorted[:11].sum())
_top_11_min_turnover = float(2.0 * (1.0 - _top_11_mass))
display(pd.Series({'gross_turnover_limit': _limit, 'required_selected_current_mass': _required_mass, 'minimum_whole_support_assets': _minimum_support, 'top_11_selected_current_mass': _top_11_mass, 'top_11_minimum_possible_turnover': _top_11_min_turnover}, name='value').to_frame())
print('release uses active-rebalance selection: inactive assets remain at current weights.')


,value
gross_turnover_limit,0.500000
required_selected_current_mass,0.750000
minimum_whole_support_assets,28.000000
top_11_selected_current_mass,0.388269
top_11_minimum_possible_turnover,1.223461


Hybrid QAOA uses active-rebalance selection: inactive assets remain at current weights.


## Primary independent QAOA configuration

In [25]:
from dataclasses import replace
MAX_QUBITS = 11
PRIMARY_SELECTION_OBJECTIVE = 'incumbent_marginal_utility'
PRIMARY_ACTIVE_CARDINALITY = 6
if FAST_MODE:
    QAOA_REPS = 1
    QAOA_SHOTS = 512
    QAOA_MAXITER = 25
    SEEDS = [20260804, 20260805, 20260806]
    TOP_QUANTUM_SAMPLES = 16
    MAX_SUBSETS_TO_REFINE = 8
else:
    QAOA_REPS = 2
    QAOA_SHOTS = 1024
    QAOA_MAXITER = 60
    SEEDS = [20260804 + offset for offset in range(20)]
    TOP_QUANTUM_SAMPLES = 30
    MAX_SUBSETS_TO_REFINE = 15
QAOA_AGGREGATION = 0.25
EXACT_CANDIDATES_TO_REFINE = 16
BASE_HYBRID_CONFIG = hybrid.QiskitHybridConfig(max_qubits=MAX_QUBITS, cardinality=PRIMARY_ACTIVE_CARDINALITY, reps=QAOA_REPS, shots=QAOA_SHOTS, maxiter=QAOA_MAXITER, seed=SEEDS[0], top_quantum_samples=TOP_QUANTUM_SAMPLES, maximum_subsets_to_refine=MAX_SUBSETS_TO_REFINE, material_incumbent_weight=0.015, scenario_proxy_share=0.35, class_balance_strength=0.0, selected_minimum_weight=0.0, exact_enumeration_limit=20000, run_numpy_exact_eigensolver=False, retain_raw_qiskit_objects=False, initial_point=None, callback_checkpoint_path=None, callback_checkpoint_interval=5, selection_mode='active_rebalance', selection_objective=PRIMARY_SELECTION_OBJECTIVE, active_balance_strength=1.0, trade_materiality_floor=0.0005, trade_materiality_fraction=0.01, marginal_transfer_size=0.0025, marginal_improvement_fraction=0.05, inferred_cardinality_cap=6, qaoa_aggregation=QAOA_AGGREGATION, transpiler_optimization_level=2, use_cardinality_preserving_mixer=True, exact_candidates_to_refine=EXACT_CANDIDATES_TO_REFINE, executed_trade_threshold=1e-05)
BASE_HYBRID_CONFIG.validate()
PREVIEW_REDUCED = hybrid.reduce_active_universe(context=PRIMARY_CONTEXT, classical_reference=PRIMARY_CLASSICAL_REFERENCE, preferences=HYBRID_PREFERENCES, scales=GOAL_SCALES, mix=step5.GOAL_MIX, config=BASE_HYBRID_CONFIG)
MARGINAL_SIGNAL_COUNT_CONFIG = replace(BASE_HYBRID_CONFIG, cardinality=None)
INFERRED_ACTIVE_CARDINALITY = hybrid.infer_active_cardinality(reduced=PREVIEW_REDUCED, config=MARGINAL_SIGNAL_COUNT_CONFIG)
_marginal_signal_values = PREVIEW_REDUCED.screening_table['marginal_improvement'].to_numpy(dtype=float)
_marginal_signal_threshold = max(1e-12, MARGINAL_SIGNAL_COUNT_CONFIG.marginal_improvement_fraction * float(np.max(_marginal_signal_values)))
RAW_QUALIFYING_MARGINAL_SIGNALS = int(np.count_nonzero(_marginal_signal_values >= _marginal_signal_threshold))
assert hybrid.infer_active_cardinality(reduced=PREVIEW_REDUCED, config=BASE_HYBRID_CONFIG) == PRIMARY_ACTIVE_CARDINALITY
print('Primary selection objective:', PRIMARY_SELECTION_OBJECTIVE)
print('Qubits:', BASE_HYBRID_CONFIG.max_qubits)
print('Raw qualifying marginal signals:', RAW_QUALIFYING_MARGINAL_SIGNALS)
print('Inferred active cardinality after cap:', INFERRED_ACTIVE_CARDINALITY)
print('Pre-registered active cardinality used:', PRIMARY_ACTIVE_CARDINALITY)
print('QAOA seeds:', len(SEEDS))
print('QAOA depth / shots / maxiter:', QAOA_REPS, QAOA_SHOTS, QAOA_MAXITER)
display(PREVIEW_REDUCED.screening_table[['marginal_direction', 'marginal_improvement', 'marginal_counterparty', 'selection_trade', 'active_selection_score', 'qubit_index']].sort_values('active_selection_score', ascending=False))
BASE_HYBRID_CONFIG


Primary selection objective: incumbent_marginal_utility
Qubits: 11
Raw qualifying marginal signals: 11
Inferred active cardinality after cap: 6
Pre-registered active cardinality used: 6
QAOA seeds: 20
QAOA depth / shots / maxiter: 2 1024 60


,marginal_direction,marginal_improvement,marginal_counterparty,selection_trade,active_selection_score,qubit_index
ticker,,,,,,
SGOV,1,0.002695,QQQ,0.002500,0.999769,0
QQQ,-1,0.002695,SGOV,-0.002500,0.945708,3
VUG,-1,0.002370,SGOV,-0.002198,0.846189,4
BIL,1,0.002286,QQQ,0.002121,0.840854,1
MTUM,-1,0.002196,SGOV,-0.002037,0.780136,5
SPY,-1,0.002061,SGOV,-0.001912,0.738301,6
EWC,-1,0.002111,SGOV,-0.001959,0.726455,7
EEM,-1,0.002084,SGOV,-0.001933,0.701384,8
VGK,-1,0.001994,SGOV,-0.001850,0.698701,9


QiskitHybridConfig(max_qubits=11, cardinality=6, reps=2, shots=1024, maxiter=60, seed=20260804, top_quantum_samples=30, maximum_subsets_to_refine=15, material_incumbent_weight=0.015, scenario_proxy_share=0.35, class_balance_strength=0.0, selected_minimum_weight=0.0, exact_enumeration_limit=20000, run_numpy_exact_eigensolver=False, retain_raw_qiskit_objects=False, initial_point=None, callback_checkpoint_path=None, callback_checkpoint_interval=5, selection_mode='active_rebalance', selection_objective='incumbent_marginal_utility', active_balance_strength=1.0, trade_materiality_floor=0.0005, trade_materiality_fraction=0.01, marginal_transfer_size=0.0025, marginal_improvement_fraction=0.05, inferred_cardinality_cap=6, qaoa_aggregation=0.25, transpiler_optimization_level=2, use_cardinality_preserving_mixer=True, exact_candidates_to_refine=16, executed_trade_threshold=1e-05)

In [26]:
import inspect
required = ['QiskitHybridConfig', 'run_hybrid_qiskit_pipeline', 'build_marginal_utility_screening_table', 'build_active_screening_table', 'infer_active_cardinality', 'build_fixed_cardinality_qaoa_components', 'evaluate_classical_subset_baselines']
missing = [name for name in required if not hasattr(hybrid, name)]
if missing:
    raise AssertionError('Missing release functions: ' + ', '.join(missing))
source_check = inspect.getsource(hybrid.run_qiskit_qaoa)
for marker in ['AerSamplerV2', 'generate_preset_pass_manager', 'aggregation', 'build_fixed_cardinality_qaoa_components', 'feasible_probability_mass']:
    if marker not in source_check:
        raise AssertionError(f'Missing runtime marker: {marker}')
print('Compatibility check passed: independent marginal utility, target recovery, matched baselines, Aer transpilation, CVaR, fixed-cardinality mixer, and probability diagnostics are active.')


Compatibility check passed: independent marginal utility, target recovery, matched baselines, Aer transpilation, CVaR, fixed-cardinality mixer, and probability diagnostics are active.


## Aer transpilation smoke test

### Verify the binary-selection model interface

In [27]:
import inspect
_binary_model_signature = inspect.signature(hybrid.BinarySelectionModel)
print('BinarySelectionModel signature:', _binary_model_signature)
_expected_fields = {'tickers', 'Q', 'linear', 'constant', 'cardinality', 'target_class_counts'}
_actual_fields = set(_binary_model_signature.parameters)
if _actual_fields != _expected_fields:
    raise AssertionError(f'Unexpected BinarySelectionModel constructor fields. Expected {_expected_fields}, got {_actual_fields}.')
print('PASS: BinarySelectionModel constructor matches the smoke test.')


BinarySelectionModel signature: (tickers: 'list[str]', Q: 'np.ndarray', linear: 'np.ndarray', constant: 'float', cardinality: 'int', target_class_counts: 'dict[str, int]') -> None
PASS: BinarySelectionModel constructor matches the smoke test.


In [28]:
import numpy as np
_smoke_model = hybrid.BinarySelectionModel(tickers=['x0', 'x1'], Q=np.array([[0.0, 0.125], [0.125, 0.0]], dtype=float), linear=np.array([-1.0, -0.5], dtype=float), constant=0.0, cardinality=1, target_class_counts={})
_smoke_problem = hybrid.build_qiskit_quadratic_program(_smoke_model)
_smoke_config = hybrid.QiskitHybridConfig(max_qubits=4, cardinality=1, reps=1, shots=128, maxiter=4, seed=7, top_quantum_samples=2, maximum_subsets_to_refine=1, exact_enumeration_limit=10, run_numpy_exact_eigensolver=False, retain_raw_qiskit_objects=False, callback_checkpoint_path=None, selection_mode='active_rebalance', qaoa_aggregation=0.25, use_cardinality_preserving_mixer=True, transpiler_optimization_level=2)
_smoke_result = hybrid.run_qiskit_qaoa(quadratic_program=_smoke_problem, model=_smoke_model, config=_smoke_config)
if _smoke_result['callback_history'].empty:
    raise AssertionError('QAOA produced no callback evaluations.')
if _smoke_result['samples'].empty:
    raise AssertionError('QAOA produced no fixed-cardinality sample.')
print('PASS: Aer executed the transpiled fixed-cardinality QAOA smoke test.')
print('QAOA transpilation keyword:', _smoke_result['transpiler_keyword'])
display(_smoke_result['samples'].head())


PASS: Aer executed the transpiled fixed-cardinality QAOA smoke test.
QAOA transpilation keyword: pass_manager


,bitstring,cardinality,raw_solver_probability,reported_objective,economic_energy,status,selected_indices,selected_tickers,conditional_probability,probability_is_near_uniform
0,10,1,0.578125,-1.0,-1.0,OptimizationResultStatus.SUCCESS,"(0,)",x0,0.578125,False
1,01,1,0.421875,-0.5,-0.5,OptimizationResultStatus.SUCCESS,"(1,)",x1,0.421875,False


## Run or resume short QAOA seeds

## release pre-seed integrity guard

In [29]:
import inspect
from pathlib import Path

if hybrid.__name__ != "step_05q_hybrid_qaoa_final":
    raise RuntimeError(
        f"Wrong module loaded: {hybrid.__name__}"
    )

_module_path = Path(
    hybrid.__file__
).resolve()

if _module_path.name != (
    "step_05q_hybrid_qaoa_final.py"
):
    raise RuntimeError(
        f"Wrong module file: {_module_path}"
    )

pipeline_source = inspect.getsource(
    hybrid.run_hybrid_qiskit_pipeline
)
qaoa_source = inspect.getsource(
    hybrid.run_qiskit_qaoa
)

for marker in [
    "exact_active_set_benchmark",
    "best_additional_exact_profile",
    "selection_objective",
]:
    if marker not in pipeline_source:
        raise RuntimeError(
            f"Missing release pipeline marker: {marker}"
        )

for marker in [
    "build_fixed_cardinality_qaoa_components",
    "qaoa_aggregation",
    "feasible_probability_mass",
]:
    if marker not in qaoa_source:
        raise RuntimeError(
            f"Missing release QAOA marker: {marker}"
        )

assert (
    BASE_HYBRID_CONFIG.selection_mode
    == "active_rebalance"
)
assert (
    BASE_HYBRID_CONFIG.selection_objective
    == "incumbent_marginal_utility"
)
assert (
    QAOA_RUN_DIR.name
    == "qaoa_seed_runs"
)
assert (
    QAOA_PROGRESS_DIR.name
    == "qaoa_progress"
)

print("PASS: release pre-seed integrity guard.")
print("Module:", _module_path)
print(
    "Selection objective:",
    BASE_HYBRID_CONFIG.selection_objective,
)
print(
    "Risk policy:",
    RISK_POLICY_MODE,
)
print(
    "Seed count:",
    len(SEEDS),
)

_cardinality_source = inspect.getsource(
    hybrid.infer_active_cardinality
)

for required_marker in [
    "config.cardinality is not None",
    "marginal_improvement",
    "absolute_reference_trade",
]:
    if required_marker not in _cardinality_source:
        raise RuntimeError(
            "Missing release cardinality marker: "
            f"{required_marker}"
        )

if (
    hybrid.infer_active_cardinality(
        reduced=PREVIEW_REDUCED,
        config=BASE_HYBRID_CONFIG,
    )
    != PRIMARY_ACTIVE_CARDINALITY
):
    raise RuntimeError(
        "The fixed active-cardinality budget "
        "was not respected."
    )

print(
    "Cardinality implementation:",
    "single, mode-aware, fixed-budget compatible",
)


PASS: Hybrid QAOA pre-seed integrity guard.
Module: /content/step_05q_hybrid_qaoa_final.py
Selection objective: incumbent_marginal_utility
Risk policy: soft_warning
Seed count: 20
Cardinality implementation: single, mode-aware, fixed-budget compatible


In [30]:
import cloudpickle
import gc
import gzip
import json
import os
import psutil
from dataclasses import replace
completed_runs = {}
_process = psutil.Process(os.getpid())
for position, seed in enumerate(SEEDS, start=1):
    result_file = QAOA_RUN_DIR / f'qaoa_independent_seed_{seed}.pkl.gz'
    progress_file = QAOA_PROGRESS_DIR / f'qaoa_independent_seed_{seed}_progress.json'
    if result_file.exists():
        print(f'Seed {seed} already completed; loading it.')
        with gzip.open(result_file, 'rb') as file:
            completed_runs[seed] = cloudpickle.load(file)
        continue
    initial_point = None
    if progress_file.exists():
        try:
            progress = json.loads(progress_file.read_text(encoding='utf-8'))
            saved = progress.get('parameters')
            if saved is not None and len(saved) == 2 * QAOA_REPS:
                initial_point = tuple((float(value) for value in saved))
                print(f'Seed {seed}: warm-starting from the saved callback checkpoint.')
        except Exception as error:
            print(f'Seed {seed}: progress file could not be loaded: {error}')
    if initial_point is None:
        rng = np.random.default_rng(seed)
        initial_point = tuple(rng.uniform(-np.pi, np.pi, size=2 * QAOA_REPS).astype(float))
        print(f'Seed {seed}: independent random QAOA initial point.')
    print('\n' + '=' * 72)
    print(f'Starting seed {seed} ({position}/{len(SEEDS)})')
    print('RAM before seed:', f'{_process.memory_info().rss / 1024 ** 3:.2f} GB')
    print('=' * 72)
    run_config = replace(BASE_HYBRID_CONFIG, seed=seed, initial_point=initial_point, callback_checkpoint_path=str(progress_file))
    try:
        run_result = hybrid.run_hybrid_qiskit_pipeline(context=PRIMARY_CONTEXT, preferences=HYBRID_PREFERENCES, scales=GOAL_SCALES, mix=step5.GOAL_MIX, classical_reference=PRIMARY_CLASSICAL_REFERENCE, config=run_config)
    except Exception as error:
        print('Seed failed with module:', Path(hybrid.__file__).resolve())
        print('Selection mode:', run_config.selection_mode)
        print('Seed directory:', QAOA_RUN_DIR)
        raise
    compact = {'seed': seed, 'config': run_config, 'samples': run_result['qiskit_run']['samples'], 'callback_history': run_result['qiskit_run']['callback_history'], 'refinement_table': run_result['refinement_table'], 'hybrid_profile_result': run_result['hybrid_profile_result'], 'exact_active_profile_result': run_result['exact_active_profile_result'], 'best_available_profile_result': run_result['best_available_profile_result'], 'best_additional_exact_profile_result': run_result['best_additional_exact_profile_result'], 'exact_enumeration': run_result['exact_enumeration'], 'reduced_universe': run_result['reduced_universe'], 'selection_model': run_result['selection_model'], 'cardinality': run_result['cardinality'], 'exact_energy': run_result['qiskit_run']['exact_energy'], 'optimizer_time': run_result['qiskit_run']['optimizer_time'], 'optimizer_evaluations': run_result['qiskit_run']['optimizer_evaluations'], 'feasible_probability_mass': run_result['qiskit_run']['feasible_probability_mass'], 'mixer_type': run_result['qiskit_run']['mixer_type'], 'aggregation': run_result['qiskit_run']['aggregation'], 'qaoa_api_family': run_result['qiskit_run']['qaoa_api_family'], 'selection_mode': run_result['selection_mode'], 'selection_objective': run_result['selection_objective'], 'selection_source': run_result['selection_source'], 'qaoa_feasible_candidate_found': run_result['qaoa_feasible_candidate_found'], 'transpiler_keyword': run_result['qiskit_run']['transpiler_keyword'], 'transpiler_optimization_level': run_result['qiskit_run']['transpiler_optimization_level']}
    temporary = result_file.with_suffix('.tmp')
    with gzip.open(temporary, 'wb') as file:
        cloudpickle.dump(compact, file)
    temporary.replace(result_file)
    completed_runs[seed] = compact
    print('Saved:', result_file)
    print('RAM after seed:', f'{_process.memory_info().rss / 1024 ** 3:.2f} GB')
    if AUTO_DOWNLOAD_BUNDLES:
        LATEST_RESUME_BUNDLE = create_resume_bundle(f'completed_seed_{seed}', download=True)
    del run_result
    gc.collect()
print('\nCompleted seeds:', sorted(completed_runs))


Seed 20260804: independent random QAOA initial point.

Starting seed 20260804 (1/20)
RAM before seed: 0.33 GB
Saved: /content/quantum_portfolio_checkpoints/qaoa_seed_runs/qaoa_independent_seed_20260804.pkl.gz
RAM after seed: 0.33 GB
Created resume bundle: /content/quantum_portfolio_resume_bundle_completed_seed_20260804_20260805T174137Z.zip
Size: 0.54 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Seed 20260805: independent random QAOA initial point.

Starting seed 20260805 (2/20)
RAM before seed: 0.33 GB
Saved: /content/quantum_portfolio_checkpoints/qaoa_seed_runs/qaoa_independent_seed_20260805.pkl.gz
RAM after seed: 0.33 GB
Created resume bundle: /content/quantum_portfolio_resume_bundle_completed_seed_20260805_20260805T174154Z.zip
Size: 0.57 MB


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Seed 20260823: independent random QAOA initial point.

Starting seed 20260823 (20/20)
RAM before seed: 0.34 GB
Saved: /content/quantum_portfolio_checkpoints/qaoa_seed_runs/qaoa_independent_seed_20260823.pkl.gz
RAM after seed: 0.34 GB
Created resume bundle: /content/quantum_portfolio_resume_bundle_completed_seed_20260823_20260805T174434Z.zip
Size: 1.05 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Completed seeds: [20260804, 20260805, 20260806, 20260807, 20260808, 20260809, 20260810, 20260811, 20260812, 20260813, 20260814, 20260815, 20260816, 20260817, 20260818, 20260819, 20260820, 20260821, 20260822, 20260823]


## Select the best completed seed

In [31]:
import pandas as pd
if not completed_runs:
    raise RuntimeError('No QAOA seed completed.')
records = []
for seed, result in completed_runs.items():
    profile = result['hybrid_profile_result']
    samples = result['samples']
    exact_profile = result.get('exact_active_profile_result')
    exact_bitstring = str(result['exact_enumeration'].iloc[0]['bitstring'])
    measured_bitstrings = set(samples['bitstring'].astype(str))
    exact_hit = exact_bitstring in measured_bitstrings
    best_energy = float(samples['economic_energy'].min())
    exact_energy = float(result['exact_energy'])
    records.append({'seed': seed, 'normalized_objective': profile['hybrid_normalized_objective'], 'expected_total_return': profile['metrics']['expected_total_return'], 'volatility': profile['metrics']['volatility'], 'worst_scenario_loss': profile['metrics']['worst_scenario_loss'], 'gross_turnover': profile['metrics']['gross_turnover'], 'total_trading_cost': profile['metrics']['total_trading_cost'], 'active_candidates': len(profile['selected_tickers']), 'executed_trades': profile.get('executed_trade_count', np.nan), 'best_measured_qubo_energy': best_energy, 'exact_qubo_energy': exact_energy, 'qubo_energy_gap': best_energy - exact_energy, 'exact_optimum_sampled': exact_hit, 'feasible_probability_mass': result['feasible_probability_mass'], 'optimizer_time_seconds': result['optimizer_time'], 'exact_active_objective': exact_profile['hybrid_normalized_objective'] if exact_profile is not None else np.nan})
QAOA_SEED_SUMMARY = pd.DataFrame(records).sort_values(['normalized_objective', 'qubo_energy_gap']).reset_index(drop=True)
display(QAOA_SEED_SUMMARY.style.format({'normalized_objective': '{:.6f}', 'expected_total_return': '{:.2%}', 'volatility': '{:.2%}', 'worst_scenario_loss': '{:.2%}', 'gross_turnover': '{:.2%}', 'total_trading_cost': '{:.4%}', 'active_candidates': '{:.0f}', 'executed_trades': '{:.0f}', 'best_measured_qubo_energy': '{:.6f}', 'exact_qubo_energy': '{:.6f}', 'qubo_energy_gap': '{:.6f}', 'feasible_probability_mass': '{:.2%}', 'optimizer_time_seconds': '{:.2f}', 'exact_active_objective': '{:.6f}'}))
QAOA_RELIABILITY_SUMMARY = pd.Series({'seed_count': len(QAOA_SEED_SUMMARY), 'exact_optimum_hit_rate': QAOA_SEED_SUMMARY['exact_optimum_sampled'].mean(), 'median_qubo_energy_gap': QAOA_SEED_SUMMARY['qubo_energy_gap'].median(), 'worst_qubo_energy_gap': QAOA_SEED_SUMMARY['qubo_energy_gap'].max(), 'median_financial_objective': QAOA_SEED_SUMMARY['normalized_objective'].median(), 'worst_financial_objective': QAOA_SEED_SUMMARY['normalized_objective'].max(), 'median_runtime_seconds': QAOA_SEED_SUMMARY['optimizer_time_seconds'].median()}, name='value')
display(QAOA_RELIABILITY_SUMMARY.to_frame())
best_seed = int(QAOA_SEED_SUMMARY.iloc[0]['seed'])
BEST_QAOA_RUN = completed_runs[best_seed]
HYBRID_RESULT = {'reduced_universe': BEST_QAOA_RUN['reduced_universe'], 'selection_model': BEST_QAOA_RUN['selection_model'], 'exact_enumeration': BEST_QAOA_RUN['exact_enumeration'], 'qiskit_run': {'samples': BEST_QAOA_RUN['samples'], 'callback_history': BEST_QAOA_RUN['callback_history'], 'exact_energy': BEST_QAOA_RUN['exact_energy'], 'optimizer_time': BEST_QAOA_RUN['optimizer_time'], 'optimizer_evaluations': BEST_QAOA_RUN['optimizer_evaluations'], 'feasible_probability_mass': BEST_QAOA_RUN['feasible_probability_mass'], 'mixer_type': BEST_QAOA_RUN['mixer_type'], 'aggregation': BEST_QAOA_RUN['aggregation'], 'qaoa_api_family': BEST_QAOA_RUN['qaoa_api_family']}, 'refinement_table': BEST_QAOA_RUN['refinement_table'], 'hybrid_profile_result': BEST_QAOA_RUN['hybrid_profile_result'], 'exact_active_profile_result': BEST_QAOA_RUN['exact_active_profile_result'], 'best_additional_exact_profile_result': BEST_QAOA_RUN.get('best_additional_exact_profile_result'), 'best_available_profile_result': BEST_QAOA_RUN['best_available_profile_result'], 'cardinality': BEST_QAOA_RUN['cardinality'], 'selection_mode': BEST_QAOA_RUN.get('selection_mode', 'active_rebalance'), 'selection_objective': BEST_QAOA_RUN.get('selection_objective', PRIMARY_SELECTION_OBJECTIVE), 'selection_source': BEST_QAOA_RUN.get('selection_source', 'qaoa')}
HYBRID_QAOA_PROFILE_RESULT = HYBRID_RESULT['hybrid_profile_result']
EXACT_ACTIVE_PROFILE_RESULT = HYBRID_RESULT['exact_active_profile_result']
BEST_ADDITIONAL_EXACT_PROFILE_RESULT = HYBRID_RESULT['best_additional_exact_profile_result']
BEST_HYBRID_AVAILABLE_PROFILE_RESULT = HYBRID_RESULT['best_available_profile_result']
HYBRID_CONFIG = BEST_QAOA_RUN['config']
print('Best QAOA seed:', best_seed)
print('Primary selection objective:', HYBRID_RESULT['selection_objective'])
print('QAOA active candidates:', HYBRID_QAOA_PROFILE_RESULT['selected_tickers'])
print('QAOA executed trades:', HYBRID_QAOA_PROFILE_RESULT.get('executed_trade_tickers', []))
print('Exact benchmark objective:', EXACT_ACTIVE_PROFILE_RESULT['hybrid_normalized_objective'] if EXACT_ACTIVE_PROFILE_RESULT is not None else None)


,seed,normalized_objective,expected_total_return,volatility,worst_scenario_loss,gross_turnover,total_trading_cost,active_candidates,executed_trades,best_measured_qubo_energy,exact_qubo_energy,qubo_energy_gap,exact_optimum_sampled,feasible_probability_mass,optimizer_time_seconds,exact_active_objective
0,20260804,-0.798818,5.45%,7.87%,12.72%,11.06%,0.0015%,6,4,-4.067965,-4.067965,0.000000,True,100.00%,4.82,-0.798818
1,20260805,-0.798818,5.45%,7.87%,12.72%,11.06%,0.0015%,6,4,-4.067965,-4.067965,0.000000,True,100.00%,3.63,-0.798818
2,20260807,-0.798818,5.45%,7.87%,12.72%,11.06%,0.0015%,6,4,-4.067965,-4.067965,0.000000,True,100.00%,3.99,-0.798818
3,20260808,-0.798818,5.45%,7.87%,12.72%,11.06%,0.0015%,6,4,-4.067965,-4.067965,0.000000,True,100.00%,3.05,-0.798818
4,20260810,-0.798818,5.45%,7.87%,12.72%,11.06%,0.0015%,6,4,-4.067965,-4.067965,0.000000,True,100.00%,1.43,-0.798818
5,20260811,-0.798818,5.45%,7.87%,12.72%,11.06%,0.0015%,6,4,-4.067965,-4.067965,0.000000,True,100.00%,3.08,-0.798818
6,20260812,-0.798818,5.45%,7.87%,12.72%,11.06%,0.0015%,6,4,-4.067965,-4.067965,0.000000,True,100.00%,1.78,-0.798818
7,20260814,-0.798818,5.45%,7.87%,12.72%,11.06%,0.0015%,6,4,-4.067965,-4.067965,0.000000,True,100.00%,1.95,-0.798818
8,20260815,-0.798818,5.45%,7.87%,12.72%,11.06%,0.0015%,6,4,-4.067965,-4.067965,0.000000,True,100.00%,2.54,-0.798818
9,20260816,-0.798818,5.45%,7.87%,12.72%,11.06%,0.0015%,6,4,-4.067965,-4.067965,0.000000,True,100.00%,1.46,-0.798818


,value
seed_count,20.000000
exact_optimum_hit_rate,0.750000
median_qubo_energy_gap,0.000000
worst_qubo_energy_gap,0.083637
median_financial_objective,-0.798818
worst_financial_objective,-0.798047
median_runtime_seconds,1.918267


Best QAOA seed: 20260804
Primary selection objective: incumbent_marginal_utility
QAOA active candidates: ['QQQ', 'VUG', 'MTUM', 'SHY', 'BIL', 'SGOV']
QAOA executed trades: ['QQQ', 'VUG', 'MTUM', 'SGOV']
Exact benchmark objective: -0.7988178089556922


# Step 5Q-C2 — Separate classical-target-recovery audit

In [32]:
TARGET_RECOVERY_CONFIG = replace(BASE_HYBRID_CONFIG, selection_objective='classical_target_recovery', cardinality=None, seed=20260901)
TARGET_RECOVERY_REDUCED = hybrid.reduce_active_universe(context=PRIMARY_CONTEXT, classical_reference=PRIMARY_CLASSICAL_REFERENCE, preferences=HYBRID_PREFERENCES, scales=GOAL_SCALES, mix=step5.GOAL_MIX, config=TARGET_RECOVERY_CONFIG)
TARGET_RECOVERY_CARDINALITY = hybrid.infer_active_cardinality(reduced=TARGET_RECOVERY_REDUCED, config=TARGET_RECOVERY_CONFIG)
TARGET_RECOVERY_MODEL = hybrid.build_active_selection_model(context=PRIMARY_CONTEXT, classical_reference=PRIMARY_CLASSICAL_REFERENCE, reduced=TARGET_RECOVERY_REDUCED, cardinality=TARGET_RECOVERY_CARDINALITY, config=TARGET_RECOVERY_CONFIG)
TARGET_RECOVERY_ENUMERATION = hybrid.enumerate_fixed_cardinality(TARGET_RECOVERY_MODEL, TARGET_RECOVERY_CONFIG.exact_enumeration_limit)
_target_bitstring = str(TARGET_RECOVERY_ENUMERATION.iloc[0]['bitstring'])
_target_bits = np.fromiter((int(value) for value in _target_bitstring), dtype=int)
TARGET_RECOVERY_EXACT_PROFILE = hybrid.refine_active_subset(context=PRIMARY_CONTEXT, classical_reference=PRIMARY_CLASSICAL_REFERENCE, reduced=TARGET_RECOVERY_REDUCED, bits=_target_bits, preferences=HYBRID_PREFERENCES, scales=GOAL_SCALES, mix=step5.GOAL_MIX, config=TARGET_RECOVERY_CONFIG, label='Exact classical-target-recovery benchmark')
TARGET_RECOVERY_AUDIT = pd.Series({'selection_objective': 'classical_target_recovery', 'qubits': TARGET_RECOVERY_REDUCED.n_qubits, 'cardinality': TARGET_RECOVERY_CARDINALITY, 'exact_qubo_energy': float(TARGET_RECOVERY_ENUMERATION.iloc[0]['energy']), 'continuous_objective': TARGET_RECOVERY_EXACT_PROFILE['hybrid_normalized_objective'], 'classical_objective': hybrid.normalized_step5_objective(context=PRIMARY_CONTEXT, weights=PRIMARY_CLASSICAL_REFERENCE['result'].weights, preferences=HYBRID_PREFERENCES, scales=GOAL_SCALES, mix=step5.GOAL_MIX)['normalized_objective'], 'executed_trades': TARGET_RECOVERY_EXACT_PROFILE['executed_trade_count']}, name='value')
display(TARGET_RECOVERY_REDUCED.screening_table[['reference_trade', 'absolute_reference_trade', 'active_selection_score', 'qubit_index']].sort_values('absolute_reference_trade', ascending=False))
display(TARGET_RECOVERY_ENUMERATION.head(10))
display(TARGET_RECOVERY_AUDIT.to_frame())
print('Target-recovery selected assets:', TARGET_RECOVERY_EXACT_PROFILE['selected_tickers'])


,reference_trade,absolute_reference_trade,active_selection_score,qubit_index
ticker,,,,
SGOV,6.545693e-02,6.545693e-02,9.997693e-01,0
VUG,-2.371467e-02,2.371467e-02,3.486356e-01,1
QQQ,-2.056527e-02,2.056527e-02,2.971228e-01,2
MTUM,-1.117699e-02,1.117699e-02,1.634671e-01,3
FXE,-5.097021e-03,5.097021e-03,7.233815e-02,4
UUP,-4.902979e-03,4.902979e-03,6.975765e-02,5
BWX,-8.382184e-15,8.382184e-15,1.265478e-13,6
DBMF,-8.321469e-15,8.321469e-15,1.184874e-13,8
MUB,-8.049117e-15,8.049117e-15,1.200941e-13,7


,bitstring,energy,selected_indices,selected_tickers
0,11111100000,-1.212018,"(0, 1, 2, 3, 4, 5)","SGOV, VUG, QQQ, MTUM, FXE, UUP"
1,11110110000,-1.202969,"(0, 1, 2, 3, 5, 6)","SGOV, VUG, QQQ, MTUM, UUP, BWX"
2,11110101000,-1.202969,"(0, 1, 2, 3, 5, 7)","SGOV, VUG, QQQ, MTUM, UUP, MUB"
3,11110100100,-1.202969,"(0, 1, 2, 3, 5, 8)","SGOV, VUG, QQQ, MTUM, UUP, DBMF"
4,11110100010,-1.202969,"(0, 1, 2, 3, 5, 9)","SGOV, VUG, QQQ, MTUM, UUP, GLD"
5,11110100001,-1.202969,"(0, 1, 2, 3, 5, 10)","SGOV, VUG, QQQ, MTUM, UUP, SPY"
6,11111010000,-1.202322,"(0, 1, 2, 3, 4, 6)","SGOV, VUG, QQQ, MTUM, FXE, BWX"
7,11111001000,-1.202322,"(0, 1, 2, 3, 4, 7)","SGOV, VUG, QQQ, MTUM, FXE, MUB"
8,11111000100,-1.202322,"(0, 1, 2, 3, 4, 8)","SGOV, VUG, QQQ, MTUM, FXE, DBMF"
9,11111000010,-1.202322,"(0, 1, 2, 3, 4, 9)","SGOV, VUG, QQQ, MTUM, FXE, GLD"


,value
selection_objective,classical_target_recovery
qubits,11
cardinality,6
exact_qubo_energy,-1.212018
continuous_objective,-0.799788
classical_objective,-0.799788
executed_trades,6


Target-recovery selected assets: ['QQQ', 'VUG', 'MTUM', 'SGOV', 'UUP', 'FXE']


# Step 5Q-C3 — Matched classical subset-selection baselines

In [33]:
CLASSICAL_SUBSET_BASELINE_TABLE, CLASSICAL_SUBSET_BASELINE_PROFILES = hybrid.evaluate_classical_subset_baselines(context=PRIMARY_CONTEXT, classical_reference=PRIMARY_CLASSICAL_REFERENCE, reduced=HYBRID_RESULT['reduced_universe'], model=HYBRID_RESULT['selection_model'], preferences=HYBRID_PREFERENCES, scales=GOAL_SCALES, mix=step5.GOAL_MIX, config=HYBRID_CONFIG, random_budget=20, seed=20260902)
successful_baselines = CLASSICAL_SUBSET_BASELINE_TABLE.loc[CLASSICAL_SUBSET_BASELINE_TABLE['success']].copy()
display(successful_baselines.sort_values(['normalized_objective', 'qubo_energy']).style.format({'qubo_energy': '{:.6f}', 'normalized_objective': '{:.6f}', 'expected_total_return': '{:.2%}', 'volatility': '{:.2%}', 'worst_scenario_loss': '{:.2%}', 'gross_turnover': '{:.2%}', 'total_trading_cost': '{:.4%}'}))

def _best_baseline_row(rows: pd.DataFrame) -> pd.Series:
    if rows.empty:
        raise RuntimeError('No successful baseline row was available.')
    return rows.sort_values(['normalized_objective', 'qubo_energy', 'selector'], ascending=[True, True, True]).iloc[0]
random_rows = successful_baselines.loc[successful_baselines['selector'].str.startswith('random_')]
greedy_rows = successful_baselines.loc[successful_baselines['selector'] == 'greedy']
local_search_rows = successful_baselines.loc[successful_baselines['selector'] == 'local_search']
random_best_row = _best_baseline_row(random_rows)
greedy_best_row = _best_baseline_row(greedy_rows)
local_search_best_row = _best_baseline_row(local_search_rows)
CLASSICAL_BASELINE_SUMMARY = pd.DataFrame([{'selector': 'random_best', 'source_selector': random_best_row['selector'], 'selected_tickers': random_best_row['selected_tickers'], 'normalized_objective': float(random_best_row['normalized_objective']), 'qubo_energy': float(random_best_row['qubo_energy'])}, {'selector': 'greedy', 'source_selector': greedy_best_row['selector'], 'selected_tickers': greedy_best_row['selected_tickers'], 'normalized_objective': float(greedy_best_row['normalized_objective']), 'qubo_energy': float(greedy_best_row['qubo_energy'])}, {'selector': 'local_search', 'source_selector': local_search_best_row['selector'], 'selected_tickers': local_search_best_row['selected_tickers'], 'normalized_objective': float(local_search_best_row['normalized_objective']), 'qubo_energy': float(local_search_best_row['qubo_energy'])}, {'selector': 'qaoa', 'source_selector': 'qaoa', 'selected_tickers': ', '.join(HYBRID_QAOA_PROFILE_RESULT['selected_tickers']), 'normalized_objective': float(HYBRID_QAOA_PROFILE_RESULT['hybrid_normalized_objective']), 'qubo_energy': float(HYBRID_RESULT['qiskit_run']['samples']['economic_energy'].min())}, {'selector': 'exact_enumeration', 'source_selector': 'exact_enumeration', 'selected_tickers': ', '.join(EXACT_ACTIVE_PROFILE_RESULT['selected_tickers']), 'normalized_objective': float(EXACT_ACTIVE_PROFILE_RESULT['hybrid_normalized_objective']), 'qubo_energy': float(HYBRID_RESULT['exact_enumeration'].iloc[0]['energy'])}])
display(CLASSICAL_BASELINE_SUMMARY.style.format({'normalized_objective': '{:.6f}', 'qubo_energy': '{:.6f}'}))
assert CLASSICAL_BASELINE_SUMMARY.loc[CLASSICAL_BASELINE_SUMMARY['selector'] == 'random_best', 'source_selector'].iloc[0] == random_best_row['selector']


,selector,success,bitstring,qubo_energy,normalized_objective,expected_total_return,volatility,worst_scenario_loss,gross_turnover,total_trading_cost,selected_tickers
0,greedy,True,11111100000,-4.067965,-0.798818,5.45%,7.87%,12.72%,11.06%,0.0015%,"QQQ, VUG, MTUM, SHY, BIL, SGOV"
1,local_search,True,11111100000,-4.067965,-0.798818,5.45%,7.87%,12.72%,11.06%,0.0015%,"QQQ, VUG, MTUM, SHY, BIL, SGOV"
20,random_018,True,10011001110,6.470837,-0.798047,5.47%,7.97%,12.83%,9.84%,0.0014%,"QQQ, VUG, VGK, EWC, EEM, SGOV"
3,random_001,True,10011000111,5.904920,-0.797907,5.46%,8.00%,12.85%,9.69%,0.0013%,"QQQ, VUG, USMV, VGK, EEM, SGOV"
11,random_009,True,11011000110,-1.561373,-0.797865,5.47%,8.00%,12.87%,9.48%,0.0013%,"QQQ, VUG, VGK, EEM, BIL, SGOV"
2,random_000,True,10110110010,-0.852202,-0.796502,5.46%,7.92%,12.78%,10.46%,0.0015%,"SPY, QQQ, MTUM, VGK, SHY, SGOV"
14,random_012,True,10110101001,-0.909431,-0.796172,5.47%,7.98%,12.83%,9.78%,0.0015%,"QQQ, USMV, MTUM, EWC, SHY, SGOV"
6,random_004,True,11010010110,-1.749557,-0.794931,5.46%,8.02%,12.86%,9.53%,0.0014%,"SPY, QQQ, VGK, EEM, BIL, SGOV"
10,random_008,True,10101011001,-1.247258,-0.794090,5.44%,7.98%,12.79%,10.30%,0.0014%,"SPY, VUG, USMV, EWC, SHY, SGOV"
7,random_005,True,10010001111,5.516830,-0.793701,5.48%,8.09%,12.86%,9.33%,0.0015%,"QQQ, USMV, VGK, EWC, EEM, SGOV"


,selector,source_selector,selected_tickers,normalized_objective,qubo_energy
0,random_best,random_018,"QQQ, VUG, VGK, EWC, EEM, SGOV",-0.798047,6.470837
1,greedy,greedy,"QQQ, VUG, MTUM, SHY, BIL, SGOV",-0.798818,-4.067965
2,local_search,local_search,"QQQ, VUG, MTUM, SHY, BIL, SGOV",-0.798818,-4.067965
3,qaoa,qaoa,"QQQ, VUG, MTUM, SHY, BIL, SGOV",-0.798818,-4.067965
4,exact_enumeration,exact_enumeration,"QQQ, VUG, MTUM, SHY, BIL, SGOV",-0.798818,-4.067965


# Step 5Q-C4 — Institutional implementation-cost sensitivity

In [34]:
INSTITUTIONAL_SENSITIVITY = None
INSTITUTIONAL_SENSITIVITY_TABLE = pd.DataFrame()
if RUN_MODE == 'fresh' and RUN_INSTITUTIONAL_SENSITIVITY:
    import shutil
    institutional_cost_file = SOURCE_DIR / f'{PREFIX}_cost_estimates_institutional_high_participation.csv'
    if not institutional_cost_file.exists():
        raise FileNotFoundError(institutional_cost_file)
    institutional_input_dir = OUTPUT_ROOT / 'step4_selected_inputs' / f'{DATA_SOURCE}_institutional'
    institutional_input_dir.mkdir(parents=True, exist_ok=True)
    for suffix in ['asset_statistics.csv', 'covariance.csv']:
        shutil.copy2(SOURCE_DIR / f'{PREFIX}_{suffix}', institutional_input_dir / f'{PREFIX}_{suffix}')
    factor_file = SOURCE_DIR / f'{PREFIX}_factor_loadings.csv'
    if factor_file.exists():
        shutil.copy2(factor_file, institutional_input_dir / factor_file.name)
    shutil.copy2(institutional_cost_file, institutional_input_dir / f'{PREFIX}_cost_estimates.csv')
    institutional_data = step4.load_step3_data(institutional_input_dir, PREFIX)
    institutional_current_weights = step4.build_strategic_current_portfolio(institutional_data)
    np.testing.assert_allclose(institutional_current_weights, current_weights, atol=1e-12)
    institutional_stages, institutional_scenarios, institutional_constraints = step4.run_constraint_ladder(institutional_data, institutional_current_weights, objective_weights, trading_config, solver_config)
    institutional_failures = [(stage.stage, stage.message) for stage in institutional_stages if not stage.success]
    if institutional_failures:
        raise RuntimeError(institutional_failures)
    institutional_context = step5.Step5Context(step4=step4, portfolio_data=institutional_data, current_weights=institutional_current_weights, stages=institutional_stages, scenarios=institutional_scenarios, constraints=institutional_constraints, trading_config=trading_config, daily_returns=STEP5_DAILY_RETURNS)
    institutional_classical = step5.solve_goal_profile(context=institutional_context, profile_name='Institutional-cost classical reference', preferences=HYBRID_PREFERENCES, scales=GOAL_SCALES, mix=step5.GOAL_MIX)
    institutional_config = replace(BASE_HYBRID_CONFIG, selection_objective='incumbent_marginal_utility', cardinality=PRIMARY_ACTIVE_CARDINALITY)
    institutional_reduced = hybrid.reduce_active_universe(context=institutional_context, classical_reference=institutional_classical, preferences=HYBRID_PREFERENCES, scales=GOAL_SCALES, mix=step5.GOAL_MIX, config=institutional_config)
    institutional_cardinality = int(PRIMARY_ACTIVE_CARDINALITY)
    assert hybrid.infer_active_cardinality(reduced=institutional_reduced, config=institutional_config) == institutional_cardinality
    institutional_signal_config = replace(institutional_config, cardinality=None)
    institutional_marginal_signal_count = hybrid.infer_active_cardinality(reduced=institutional_reduced, config=institutional_signal_config)
    institutional_model = hybrid.build_active_selection_model(context=institutional_context, classical_reference=institutional_classical, reduced=institutional_reduced, cardinality=institutional_cardinality, config=institutional_config)
    institutional_enumeration = hybrid.enumerate_fixed_cardinality(institutional_model, institutional_config.exact_enumeration_limit)
    institutional_bitstring = str(institutional_enumeration.iloc[0]['bitstring'])
    institutional_bits = np.fromiter((int(value) for value in institutional_bitstring), dtype=int)
    institutional_exact_profile = hybrid.refine_active_subset(context=institutional_context, classical_reference=institutional_classical, reduced=institutional_reduced, bits=institutional_bits, preferences=HYBRID_PREFERENCES, scales=GOAL_SCALES, mix=step5.GOAL_MIX, config=institutional_config, label='Institutional-cost exact active-set benchmark')
    base_set = set(EXACT_ACTIVE_PROFILE_RESULT['selected_tickers'])
    institutional_set = set(institutional_exact_profile['selected_tickers'])
    if len(institutional_set) != PRIMARY_ACTIVE_CARDINALITY:
        raise AssertionError('Institutional sensitivity did not preserve the fixed matched active budget.')
    INSTITUTIONAL_SENSITIVITY_TABLE = pd.DataFrame([{'cost_case': 'base', 'expected_total_return': EXACT_ACTIVE_PROFILE_RESULT['metrics']['expected_total_return'], 'volatility': EXACT_ACTIVE_PROFILE_RESULT['metrics']['volatility'], 'gross_turnover': EXACT_ACTIVE_PROFILE_RESULT['metrics']['gross_turnover'], 'total_trading_cost': EXACT_ACTIVE_PROFILE_RESULT['metrics']['total_trading_cost'], 'active_cardinality': PRIMARY_ACTIVE_CARDINALITY, 'cardinality_policy': 'fixed_matched_budget', 'inferred_active_cardinality_after_cap': INFERRED_ACTIVE_CARDINALITY, 'selected_tickers': ', '.join(sorted(base_set))}, {'cost_case': 'institutional_high_participation', 'expected_total_return': institutional_exact_profile['metrics']['expected_total_return'], 'volatility': institutional_exact_profile['metrics']['volatility'], 'gross_turnover': institutional_exact_profile['metrics']['gross_turnover'], 'total_trading_cost': institutional_exact_profile['metrics']['total_trading_cost'], 'active_cardinality': institutional_cardinality, 'cardinality_policy': 'fixed_matched_budget', 'inferred_active_cardinality_after_cap': institutional_marginal_signal_count, 'selected_tickers': ', '.join(sorted(institutional_set))}])
    INSTITUTIONAL_SENSITIVITY = {'context': institutional_context, 'classical_reference': institutional_classical, 'exact_active_profile': institutional_exact_profile, 'selected_set_overlap': len(base_set & institutional_set) / max(len(base_set | institutional_set), 1)}
    display(INSTITUTIONAL_SENSITIVITY_TABLE.style.format({'expected_total_return': '{:.2%}', 'volatility': '{:.2%}', 'gross_turnover': '{:.2%}', 'total_trading_cost': '{:.4%}', 'active_cardinality': '{:.0f}'}))
    print('Active-set Jaccard overlap:', INSTITUTIONAL_SENSITIVITY['selected_set_overlap'])
else:
    print('Institutional sensitivity skipped. Run the notebook in fresh mode with RUN_INSTITUTIONAL_SENSITIVITY=True to generate it.')


,cost_case,expected_total_return,volatility,gross_turnover,total_trading_cost,active_cardinality,cardinality_policy,inferred_active_cardinality_after_cap,selected_tickers
0,base,5.45%,7.87%,11.06%,0.0015%,6,fixed_matched_budget,6,"BIL, MTUM, QQQ, SGOV, SHY, VUG"
1,institutional_high_participation,5.46%,7.95%,10.03%,0.0034%,6,fixed_matched_budget,6,"BIL, MTUM, QQQ, SGOV, SHY, VUG"


Active-set Jaccard overlap: 1.0


# Step 5Q-C5 — Repeated forward synthetic robustness simulation

In [35]:
FORWARD_SIMULATION_RAW = pd.DataFrame()
FORWARD_SIMULATION_SUMMARY = pd.DataFrame()
if RUN_FORWARD_SIMULATION:
    simulation_paths = 100 if FAST_MODE else 1000
    print('Forward simulation paths:', simulation_paths)
    horizon_days = 504
    degrees_of_freedom = 7
    annual_mean = np.asarray(portfolio_data.growth, dtype=float) + np.asarray(portfolio_data.income, dtype=float)
    annual_covariance = np.asarray(portfolio_data.covariance, dtype=float)
    daily_mean = annual_mean / 252.0
    daily_covariance = annual_covariance / 252.0
    eigenvalues, eigenvectors = np.linalg.eigh(0.5 * (daily_covariance + daily_covariance.T))
    daily_cholesky = eigenvectors @ np.diag(np.sqrt(np.maximum(eigenvalues, 1e-14)))
    simulation_profiles = {'Primary classical': PRIMARY_CLASSICAL_REFERENCE, 'Independent QAOA': HYBRID_QAOA_PROFILE_RESULT, 'Independent exact active set': EXACT_ACTIVE_PROFILE_RESULT}
    if STRICT_WARNING_REFERENCE is not None:
        simulation_profiles['Strict-warning classical'] = STRICT_WARNING_REFERENCE
    records = []
    for path_offset in range(simulation_paths):
        rng = np.random.default_rng(20261000 + path_offset)
        gaussian = rng.standard_normal((horizon_days, len(annual_mean)))
        chi_square = rng.chisquare(degrees_of_freedom, size=horizon_days)
        scale = np.sqrt((degrees_of_freedom - 2.0) / chi_square)[:, None]
        simulated_returns = daily_mean[None, :] + gaussian @ daily_cholesky.T * scale
        simulated_returns = np.maximum(simulated_returns, -0.95)
        for profile_name, profile in simulation_profiles.items():
            weights = np.asarray(profile['result'].weights, dtype=float)
            portfolio_path = simulated_returns @ weights
            wealth = np.cumprod(1.0 + portfolio_path)
            running_peak = np.maximum.accumulate(wealth)
            drawdown = 1.0 - wealth / running_peak
            annual_return = wealth[-1] ** (252.0 / horizon_days) - 1.0
            annual_volatility = np.std(portfolio_path, ddof=1) * np.sqrt(252.0)
            records.append({'path_seed': 20261000 + path_offset, 'profile': profile_name, 'annualized_return': annual_return, 'annualized_volatility': annual_volatility, 'maximum_drawdown': float(np.max(drawdown)), 'terminal_wealth': float(wealth[-1])})
    FORWARD_SIMULATION_RAW = pd.DataFrame(records)
    summary_records = []
    for profile_name, frame in FORWARD_SIMULATION_RAW.groupby('profile'):
        summary_records.append({'profile': profile_name, 'paths': len(frame), 'median_return': frame['annualized_return'].median(), 'return_05': frame['annualized_return'].quantile(0.05), 'median_volatility': frame['annualized_volatility'].median(), 'median_maximum_drawdown': frame['maximum_drawdown'].median(), 'drawdown_95': frame['maximum_drawdown'].quantile(0.95), 'loss_path_frequency': frame['terminal_wealth'].lt(1.0).mean()})
    FORWARD_SIMULATION_SUMMARY = pd.DataFrame(summary_records).set_index('profile')
    display(FORWARD_SIMULATION_SUMMARY.style.format({'paths': '{:.0f}', 'median_return': '{:.2%}', 'return_05': '{:.2%}', 'median_volatility': '{:.2%}', 'median_maximum_drawdown': '{:.2%}', 'drawdown_95': '{:.2%}', 'loss_path_frequency': '{:.1%}'}))
    print('Interpretation: model-based forward simulation only; not historical or out-of-sample market evidence.')
else:
    print('Forward simulation skipped.')


Forward simulation paths: 1000


,paths,median_return,return_05,median_volatility,median_maximum_drawdown,drawdown_95,loss_path_frequency
profile,,,,,,,
Independent QAOA,1000,5.70%,-3.19%,7.87%,8.49%,14.93%,14.6%
Independent exact active set,1000,5.70%,-3.19%,7.87%,8.49%,14.93%,14.6%
Primary classical,1000,5.74%,-3.15%,7.86%,8.48%,14.91%,14.1%
Strict-warning classical,1000,5.72%,-2.66%,7.51%,7.95%,13.96%,13.5%


Interpretation: model-based forward simulation only; not historical or out-of-sample market evidence.


## Optional manual backup

In [36]:
LATEST_RESUME_BUNDLE = create_resume_bundle('manual_backup', download=True)


Created resume bundle: /content/quantum_portfolio_resume_bundle_manual_backup_20260805T174446Z.zip
Size: 1.05 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Step 5Q-D — Inspect the quantum selection problem

In [37]:
screening = HYBRID_RESULT['reduced_universe'].screening_table.sort_values(['active_selection_score', 'marginal_improvement'], ascending=[False, False])
display(screening[['asset_class', 'current_weight', 'marginal_direction', 'marginal_improvement', 'marginal_counterparty', 'selection_trade', 'trade_materiality', 'implementation_burden', 'active_selection_score', 'qubit_index']])
print('Primary selection objective:', HYBRID_RESULT['selection_objective'])
print('Inferred active cardinality:', HYBRID_RESULT['cardinality'])
print('Exact independent QUBO benchmark:')
display(HYBRID_RESULT['exact_enumeration'].head(20))
print('Best QAOA samples:')
display(HYBRID_RESULT['qiskit_run']['samples'].head(20))
print('Continuous QAOA/exact refinements:')
display(HYBRID_RESULT['refinement_table'].sort_values(['success', 'normalized_continuous_objective'], ascending=[False, True]))


,asset_class,current_weight,marginal_direction,marginal_improvement,marginal_counterparty,selection_trade,trade_materiality,implementation_burden,active_selection_score,qubit_index
ticker,,,,,,,,,,
SGOV,Cash,0.014292,1,0.002695,QQQ,0.002500,1.000000,0.000382,0.999769,0
QQQ,US Equity,0.020565,-1,0.002695,SGOV,-0.002500,1.000000,0.005149,0.945708,3
VUG,US Equity,0.023715,-1,0.002370,SGOV,-0.002198,0.879341,0.003506,0.846189,4
BIL,Cash,0.015708,1,0.002286,QQQ,0.002121,0.848409,0.000239,0.840854,1
MTUM,US Equity,0.022645,-1,0.002196,SGOV,-0.002037,0.814909,0.004985,0.780136,5
SPY,US Equity,0.024840,-1,0.002061,SGOV,-0.001912,0.764904,0.004545,0.738301,6
EWC,Developed Equity,0.018943,-1,0.002111,SGOV,-0.001959,0.783505,0.013975,0.726455,7
EEM,Emerging Equity,0.009901,-1,0.002084,SGOV,-0.001933,0.773218,0.018079,0.701384,8
VGK,Developed Equity,0.019762,-1,0.001994,SGOV,-0.001850,0.739888,0.008479,0.698701,9


Primary selection objective: incumbent_marginal_utility
Inferred active cardinality: 6
Exact independent QUBO benchmark:


,bitstring,energy,selected_indices,selected_tickers
0,11111100000,-4.067965,"(0, 1, 2, 3, 4, 5)","SGOV, BIL, SHY, QQQ, VUG, MTUM"
1,11111010000,-4.035671,"(0, 1, 2, 3, 4, 6)","SGOV, BIL, SHY, QQQ, VUG, SPY"
2,11111000001,-3.991806,"(0, 1, 2, 3, 4, 10)","SGOV, BIL, SHY, QQQ, VUG, USMV"
3,11111001000,-3.984328,"(0, 1, 2, 3, 4, 7)","SGOV, BIL, SHY, QQQ, VUG, EWC"
4,11111000010,-3.979236,"(0, 1, 2, 3, 4, 9)","SGOV, BIL, SHY, QQQ, VUG, VGK"
5,11110110000,-3.944265,"(0, 1, 2, 3, 5, 6)","SGOV, BIL, SHY, QQQ, MTUM, SPY"
6,11111000100,-3.923911,"(0, 1, 2, 3, 4, 8)","SGOV, BIL, SHY, QQQ, VUG, EEM"
7,11110101000,-3.895345,"(0, 1, 2, 3, 5, 7)","SGOV, BIL, SHY, QQQ, MTUM, EWC"
8,11110100001,-3.890603,"(0, 1, 2, 3, 5, 10)","SGOV, BIL, SHY, QQQ, MTUM, USMV"
9,11110100010,-3.884264,"(0, 1, 2, 3, 5, 9)","SGOV, BIL, SHY, QQQ, MTUM, VGK"


Best QAOA samples:


,bitstring,cardinality,raw_solver_probability,reported_objective,economic_energy,status,selected_indices,selected_tickers,conditional_probability,probability_is_near_uniform
0,11111100000,6,0.000977,-4.067965,-4.067965,OptimizationResultStatus.SUCCESS,"(0, 1, 2, 3, 4, 5)","SGOV, BIL, SHY, QQQ, VUG, MTUM",0.000977,False
1,11111000001,6,0.000977,-3.991806,-3.991806,OptimizationResultStatus.SUCCESS,"(0, 1, 2, 3, 4, 10)","SGOV, BIL, SHY, QQQ, VUG, USMV",0.000977,False
2,11111001000,6,0.001953,-3.984328,-3.984328,OptimizationResultStatus.SUCCESS,"(0, 1, 2, 3, 4, 7)","SGOV, BIL, SHY, QQQ, VUG, EWC",0.001953,False
3,11111000010,6,0.002930,-3.979236,-3.979236,OptimizationResultStatus.SUCCESS,"(0, 1, 2, 3, 4, 9)","SGOV, BIL, SHY, QQQ, VUG, VGK",0.002930,False
4,11110110000,6,0.000977,-3.944265,-3.944265,OptimizationResultStatus.SUCCESS,"(0, 1, 2, 3, 5, 6)","SGOV, BIL, SHY, QQQ, MTUM, SPY",0.000977,False
5,11111000100,6,0.005859,-3.923911,-3.923911,OptimizationResultStatus.SUCCESS,"(0, 1, 2, 3, 4, 8)","SGOV, BIL, SHY, QQQ, VUG, EEM",0.005859,False
6,11110101000,6,0.000977,-3.895345,-3.895345,OptimizationResultStatus.SUCCESS,"(0, 1, 2, 3, 5, 7)","SGOV, BIL, SHY, QQQ, MTUM, EWC",0.000977,False
7,11110100001,6,0.001953,-3.890603,-3.890603,OptimizationResultStatus.SUCCESS,"(0, 1, 2, 3, 5, 10)","SGOV, BIL, SHY, QQQ, MTUM, USMV",0.001953,False
8,11110100010,6,0.008789,-3.884264,-3.884264,OptimizationResultStatus.SUCCESS,"(0, 1, 2, 3, 5, 9)","SGOV, BIL, SHY, QQQ, MTUM, VGK",0.008789,False
9,11110011000,6,0.005859,-3.852567,-3.852567,OptimizationResultStatus.SUCCESS,"(0, 1, 2, 3, 6, 7)","SGOV, BIL, SHY, QQQ, SPY, EWC",0.005859,False


Continuous QAOA/exact refinements:


,sample_rank,selection_source,bitstring,probability,qubo_energy,success,normalized_continuous_objective,expected_total_return,volatility,worst_scenario_loss,gross_turnover,total_trading_cost,selected_tickers,message
0,0,qaoa,11111100000,0.000977,-4.067965,True,-0.798818,0.054524,0.078668,0.127160,0.110552,0.000015,"QQQ, VUG, MTUM, SHY, BIL, SGOV",Optimization terminated successfully
15,1,exact_active_set_benchmark,11111010000,0.000000,-4.035671,True,-0.798330,0.054541,0.079214,0.127736,0.104820,0.000014,"SPY, QQQ, VUG, SHY, BIL, SGOV",Optimization terminated successfully
2,2,qaoa,11111001000,0.001953,-3.984328,True,-0.798047,0.054660,0.079749,0.128278,0.098439,0.000014,"QQQ, VUG, EWC, SHY, BIL, SGOV",Optimization terminated successfully
1,1,qaoa,11111000001,0.000977,-3.991806,True,-0.797907,0.054640,0.080039,0.128537,0.096851,0.000013,"QQQ, VUG, USMV, SHY, BIL, SGOV",Optimization terminated successfully
3,3,qaoa,11111000010,0.002930,-3.979236,True,-0.797865,0.054677,0.080041,0.128685,0.094760,0.000013,"QQQ, VUG, VGK, SHY, BIL, SGOV",Optimization terminated successfully
5,5,qaoa,11111000100,0.005859,-3.923911,True,-0.797763,0.054714,0.080485,0.129323,0.088902,0.000012,"QQQ, VUG, EEM, SHY, BIL, SGOV",Optimization terminated successfully
4,4,qaoa,11110110000,0.000977,-3.944265,True,-0.796502,0.054586,0.079230,0.127758,0.104600,0.000015,"SPY, QQQ, MTUM, SHY, BIL, SGOV",Optimization terminated successfully
6,6,qaoa,11110101000,0.000977,-3.895345,True,-0.796172,0.054717,0.079801,0.128325,0.097822,0.000015,"QQQ, MTUM, EWC, SHY, BIL, SGOV",Optimization terminated successfully
7,7,qaoa,11110100001,0.001953,-3.890603,True,-0.796013,0.054688,0.080096,0.128555,0.096666,0.000014,"QQQ, USMV, MTUM, SHY, BIL, SGOV",Optimization terminated successfully
8,8,qaoa,11110100010,0.008789,-3.884264,True,-0.795957,0.054732,0.080084,0.128716,0.094280,0.000014,"QQQ, MTUM, VGK, SHY, BIL, SGOV",Optimization terminated successfully


# Step 5Q-E — Quantum diagnostics

In [38]:
import matplotlib.pyplot as plt
history = HYBRID_RESULT['qiskit_run']['callback_history']
if not history.empty:
    plt.figure(figsize=(10, 5))
    plt.plot(history['evaluation'], history['mean_energy'], marker='o', markersize=3)
    plt.xlabel('Classical QAOA parameter evaluation')
    plt.ylabel('CVaR/mean QUBO energy')
    plt.title('QAOA convergence')
    plt.grid(True, alpha=0.3)
    plt.show()
samples = HYBRID_RESULT['qiskit_run']['samples'].copy()
exact_energy = float(HYBRID_RESULT['qiskit_run']['exact_energy'])
best_energy = float(samples['economic_energy'].min())
print('Exact QUBO energy:', exact_energy)
print('Best measured QAOA energy:', best_energy)
print('QAOA energy gap:', best_energy - exact_energy)
print('Fixed-cardinality probability mass:', HYBRID_RESULT['qiskit_run']['feasible_probability_mass'])
print('Mixer:', HYBRID_RESULT['qiskit_run']['mixer_type'])
print('CVaR alpha:', HYBRID_RESULT['qiskit_run']['aggregation'])
display(samples[['bitstring', 'economic_energy', 'raw_solver_probability', 'conditional_probability', 'probability_is_near_uniform']].style.format({'economic_energy': '{:.6f}', 'raw_solver_probability': '{:.6%}', 'conditional_probability': '{:.2%}'}))
if samples['probability_is_near_uniform'].all():
    print('WARNING: retained probabilities are nearly uniform; use energy and continuous quality as primary criteria.')
else:
    print('PASS: retained QAOA probabilities are non-uniform.')


Exact QUBO energy: -4.067965043181634
Best measured QAOA energy: -4.067965043181634
QAOA energy gap: 0.0
Fixed-cardinality probability mass: 1.0
Mixer: XY_ring_with_Dicke_initial_state
CVaR alpha: 0.25


,bitstring,economic_energy,raw_solver_probability,conditional_probability,probability_is_near_uniform
0,11111100000,-4.067965,0.097656%,0.10%,False
1,11111000001,-3.991806,0.097656%,0.10%,False
2,11111001000,-3.984328,0.195312%,0.20%,False
3,11111000010,-3.979236,0.292969%,0.29%,False
4,11110110000,-3.944265,0.097656%,0.10%,False
5,11111000100,-3.923911,0.585938%,0.59%,False
6,11110101000,-3.895345,0.097656%,0.10%,False
7,11110100001,-3.890603,0.195312%,0.20%,False
8,11110100010,-3.884264,0.878906%,0.88%,False
9,11110011000,-3.852567,0.585938%,0.59%,False


PASS: retained QAOA probabilities are non-uniform.


# Step 5Q-F — Compare the classical and hybrid portfolios

In [39]:
def _comparison_row(profile, label, active_candidates=np.nan, objective_context=PRIMARY_CONTEXT):
    weights = np.asarray(profile['result'].weights, dtype=float)
    incumbent = np.asarray(objective_context.current_weights, dtype=float)
    exact = hybrid.normalized_step5_objective(context=objective_context, weights=weights, preferences=HYBRID_PREFERENCES, scales=GOAL_SCALES, mix=step5.GOAL_MIX)
    return {'profile': label, **profile['metrics'], 'normalized_objective': exact['normalized_objective'], 'nonzero_holdings': int(np.count_nonzero(weights > 1e-06)), 'active_candidates': active_candidates, 'executed_trades': int(np.count_nonzero(np.abs(weights - incumbent) > 1e-05))}
comparison_rows = [_comparison_row(PRIMARY_CLASSICAL_REFERENCE, 'Primary unrestricted classical'), _comparison_row(HYBRID_QAOA_PROFILE_RESULT, 'Independent Qiskit QAOA', len(HYBRID_QAOA_PROFILE_RESULT['selected_tickers']))]
if EXACT_ACTIVE_PROFILE_RESULT is not None:
    comparison_rows.append(_comparison_row(EXACT_ACTIVE_PROFILE_RESULT, 'Independent exact active-set benchmark', len(EXACT_ACTIVE_PROFILE_RESULT['selected_tickers'])))
comparison_rows.append(_comparison_row(TARGET_RECOVERY_EXACT_PROFILE, 'Classical-target-recovery exact audit', len(TARGET_RECOVERY_EXACT_PROFILE['selected_tickers'])))
if STRICT_WARNING_REFERENCE is not None:
    comparison_rows.append(_comparison_row(STRICT_WARNING_REFERENCE, 'Classical strict-warning', objective_context=STEP5_CONTEXT))
comparison = pd.DataFrame(comparison_rows).set_index('profile')
classical_objective = float(comparison.loc['Primary unrestricted classical', 'normalized_objective'])
comparison['objective_gap_to_primary_classical'] = comparison['normalized_objective'] - classical_objective
display(comparison[['expected_total_return', 'growth', 'income', 'volatility', 'worst_scenario_loss', 'gross_turnover', 'total_trading_cost', 'effective_holdings', 'nonzero_holdings', 'active_candidates', 'executed_trades', 'normalized_objective', 'objective_gap_to_primary_classical']].style.format({'expected_total_return': '{:.2%}', 'growth': '{:.2%}', 'income': '{:.2%}', 'volatility': '{:.2%}', 'worst_scenario_loss': '{:.2%}', 'gross_turnover': '{:.2%}', 'total_trading_cost': '{:.4%}', 'effective_holdings': '{:.2f}', 'nonzero_holdings': '{:.0f}', 'active_candidates': '{:.0f}', 'executed_trades': '{:.0f}', 'normalized_objective': '{:.6f}', 'objective_gap_to_primary_classical': '{:.6f}'}))
qaoa_gap = float(comparison.loc['Independent Qiskit QAOA', 'objective_gap_to_primary_classical'])
print('Independent QAOA objective gap to primary classical:', qaoa_gap)
print('Interpretation: the independent QUBO is not allowed to use the solved classical trade vector.')
print('\nIndependent QAOA portfolio weights:')
display(HYBRID_QAOA_PROFILE_RESULT['result'].weights.sort_values(ascending=False).head(25).to_frame('weight').style.format('{:.2%}'))


,expected_total_return,growth,income,volatility,worst_scenario_loss,gross_turnover,total_trading_cost,effective_holdings,nonzero_holdings,active_candidates,executed_trades,normalized_objective,objective_gap_to_primary_classical
profile,,,,,,,,,,,,,
Primary unrestricted classical,5.49%,2.90%,2.59%,7.86%,12.71%,13.09%,0.0020%,33.73,46,nan,6,-0.799788,0.000000
Independent Qiskit QAOA,5.45%,2.90%,2.56%,7.87%,12.72%,11.06%,0.0015%,35.48,48,6,4,-0.798818,0.000970
Independent exact active-set benchmark,5.45%,2.90%,2.56%,7.87%,12.72%,11.06%,0.0015%,35.48,48,6,4,-0.798818,0.000970
Classical-target-recovery exact audit,5.49%,2.90%,2.59%,7.86%,12.71%,13.09%,0.0020%,33.73,46,6,6,-0.799788,-0.000000
Classical strict-warning,5.48%,2.82%,2.67%,7.51%,12.17%,23.88%,0.0056%,29.35,44,nan,11,-0.782684,0.017104


Independent QAOA objective gap to primary classical: 0.0009700925809347227
Interpretation: the independent QUBO is not allowed to use the solved classical trade vector.

Independent QAOA portfolio weights:


,weight
SGOV,6.96%
TIP,5.00%
BWX,3.98%
AGG,3.68%
BNDX,3.68%
BND,3.66%
USMV,3.45%
TLT,3.41%
SHY,3.40%
IEF,3.19%


## Verify the active rebalancing sleeve

In [40]:
_hybrid_weights = np.asarray(HYBRID_QAOA_PROFILE_RESULT['result'].weights, dtype=float)
_active = set(HYBRID_QAOA_PROFILE_RESULT['selected_tickers'])
_trade_table = pd.DataFrame({'current_weight': np.asarray(current_weights, dtype=float), 'hybrid_weight': _hybrid_weights}, index=portfolio_data.tickers)
_trade_table['trade'] = _trade_table['hybrid_weight'] - _trade_table['current_weight']
_trade_table['absolute_trade'] = _trade_table['trade'].abs()
_trade_table['qaoa_active_candidate'] = [t in _active for t in _trade_table.index]
_trade_table['trade_executed'] = _trade_table['absolute_trade'] > 1e-05
display(_trade_table.loc[_trade_table['qaoa_active_candidate'] | _trade_table['trade_executed']].sort_values('absolute_trade', ascending=False).style.format({'current_weight': '{:.2%}', 'hybrid_weight': '{:.2%}', 'trade': '{:+.2%}', 'absolute_trade': '{:.2%}'}))
assert float(_trade_table.loc[~_trade_table['qaoa_active_candidate'], 'absolute_trade'].max()) <= 1e-08
print('PASS: all inactive assets remained at current weights.')
print('Active candidates:', int(_trade_table['qaoa_active_candidate'].sum()))
print('Executed trades:', int(_trade_table['trade_executed'].sum()))


,current_weight,hybrid_weight,trade,absolute_trade,qaoa_active_candidate,trade_executed
SGOV,1.43%,6.96%,+5.53%,5.53%,True,True
VUG,2.37%,0.00%,-2.37%,2.37%,True,True
QQQ,2.06%,0.00%,-2.06%,2.06%,True,True
MTUM,2.26%,1.16%,-1.10%,1.10%,True,True
SHY,3.40%,3.40%,-0.00%,0.00%,True,False
BIL,1.57%,1.57%,-0.00%,0.00%,True,False


PASS: all inactive assets remained at current weights.
Active candidates: 6
Executed trades: 4


# Step 5Q-G — Verify all final portfolio constraints

### ScenarioSet field compatibility check

In [41]:
if hasattr(scenarios, 'hard_loss_limits'):
    print('Scenario hard-limit field: hard_loss_limits')
elif hasattr(scenarios, 'hard_limits'):
    print('Scenario hard-limit field: hard_limits (compatibility mode)')
else:
    raise AttributeError('No scenario hard-limit field was found.')


Scenario hard-limit field: hard_loss_limits


In [42]:
hybrid_audit = HYBRID_QAOA_PROFILE_RESULT['result'].constraint_audit.copy()
display(hybrid_audit)
failed = hybrid_audit.loc[~hybrid_audit['satisfied']]
if not failed.empty:
    display(failed)
    raise AssertionError('Hard constraints failed.')
print('PASS: all audited hard constraints are satisfied.')
scenario_set = PRIMARY_SCENARIOS
scenario_losses = scenario_set.loss_matrix @ HYBRID_QAOA_PROFILE_RESULT['result'].weights.to_numpy(dtype=float)
if hasattr(scenario_set, 'hard_loss_limits'):
    hard = np.asarray(scenario_set.hard_loss_limits, dtype=float)
elif hasattr(scenario_set, 'hard_limits'):
    hard = np.asarray(scenario_set.hard_limits, dtype=float)
else:
    raise AttributeError('No scenario hard-limit field.')
warning = np.asarray(scenario_set.warning_thresholds, dtype=float)
scenario_comparison = pd.DataFrame({'scenario_loss': scenario_losses, 'warning_threshold': warning, 'hard_limit': hard, 'warning_excess': np.maximum(scenario_losses - warning, 0.0), 'warning_satisfied': scenario_losses <= warning + 5e-06, 'hard_limit_slack': hard - scenario_losses, 'hard_limit_satisfied': scenario_losses <= hard + 5e-06}, index=scenario_set.names)
display(scenario_comparison.style.format({'scenario_loss': '{:.2%}', 'warning_threshold': '{:.2%}', 'hard_limit': '{:.2%}', 'warning_excess': '{:.2%}', 'hard_limit_slack': '{:.2%}'}))
if not scenario_comparison['hard_limit_satisfied'].all():
    raise AssertionError('A scenario hard limit failed.')
print('PASS: every scenario loss is below its hard limit.')
warning_breaches = int((~scenario_comparison['warning_satisfied']).sum())
if warning_breaches:
    print('SOFT WARNING:', warning_breaches, 'scenario warning threshold(s) exceeded.')
else:
    print('PASS: all soft warning thresholds are satisfied.')
print('Risk-policy mode:', RISK_POLICY_MODE)


,category,constraint,value,lower,upper,lower_slack,upper_slack,satisfied
0,budget,full_investment,1.000000e+00,1.000000,1.000000,-7.771561e-16,7.771561e-16,True
1,asset,minimum_weight,2.749696e-15,0.000000,inf,2.749696e-15,NaN,True
2,asset,weight_SPY,2.484007e-02,0.024840,0.024840,0.000000e+00,0.000000e+00,True
3,asset,weight_QQQ,2.749696e-15,0.000000,0.100000,2.749696e-15,1.000000e-01,True
4,asset,weight_IWM,2.014312e-02,0.020143,0.020143,0.000000e+00,0.000000e+00,True
...,...,...,...,...,...,...,...,...
127,scenario,scenario_Global equity selloff,1.141428e-01,-inf,0.154101,NaN,3.995780e-02,True
128,scenario,scenario_Inflation and rate shock,7.681432e-02,-inf,0.109719,NaN,3.290446e-02,True
129,scenario,scenario_Credit and liquidity crisis,1.040301e-01,-inf,0.139090,NaN,3.506022e-02,True
130,scenario,scenario_Commodity supply shock,3.074958e-02,-inf,0.060282,NaN,2.953263e-02,True


PASS: all audited hard constraints are satisfied.


,scenario_loss,warning_threshold,hard_limit,warning_excess,warning_satisfied,hard_limit_slack,hard_limit_satisfied
Global equity selloff,11.41%,11.41%,15.41%,0.00%,False,4.00%,True
Inflation and rate shock,7.68%,6.97%,10.97%,0.71%,False,3.29%,True
Credit and liquidity crisis,10.40%,9.91%,13.91%,0.49%,False,3.51%,True
Commodity supply shock,3.07%,2.03%,6.03%,1.05%,False,2.95%,True
Broad deleveraging shock,12.72%,12.33%,16.33%,0.39%,False,3.61%,True


PASS: every scenario loss is below its hard limit.
SOFT WARNING: 5 scenario warning threshold(s) exceeded.
Risk-policy mode: soft_warning


# Step 5Q-H — Make the hybrid portfolio available to Steps 6 and 7

In [43]:
STEP5_PROFILE_RESULTS = {'Primary unrestricted classical': PRIMARY_CLASSICAL_REFERENCE, 'Independent Qiskit QAOA': HYBRID_QAOA_PROFILE_RESULT, 'Classical-target-recovery exact audit': TARGET_RECOVERY_EXACT_PROFILE}
if EXACT_ACTIVE_PROFILE_RESULT is not None:
    STEP5_PROFILE_RESULTS['Independent exact active-set benchmark'] = EXACT_ACTIVE_PROFILE_RESULT
if STRICT_WARNING_REFERENCE is not None:
    STEP5_PROFILE_RESULTS['Classical strict-warning reference'] = STRICT_WARNING_REFERENCE
for selector_name in ['greedy', 'local_search']:
    if selector_name in CLASSICAL_SUBSET_BASELINE_PROFILES:
        STEP5_PROFILE_RESULTS[f'Classical subset baseline: {selector_name}'] = CLASSICAL_SUBSET_BASELINE_PROFILES[selector_name]
print('Profiles available for Step 6 and Step 7:')
print(list(STEP5_PROFILE_RESULTS))


Profiles available for Step 6 and Step 7:
['Primary unrestricted classical', 'Independent Qiskit QAOA', 'Classical-target-recovery exact audit', 'Independent exact active-set benchmark', 'Classical strict-warning reference', 'Classical subset baseline: greedy', 'Classical subset baseline: local_search']


# Step 5Q-I — Export and download all outputs

In [44]:
import gc
import matplotlib.pyplot as plt
plt.close('all')
gc.collect()
print('Released unused Qiskit and plotting memory before export.')


Released unused Qiskit and plotting memory before export.


In [45]:
from pathlib import Path
import importlib.metadata
import json
import shutil
from google.colab import files
OUTPUT_ROOT = Path(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
HYBRID_OUTPUT = OUTPUT_ROOT / 'step_05q_hybrid_qaoa' / DATA_SOURCE / STEP4_COST_SCENARIO
HYBRID_OUTPUT.mkdir(parents=True, exist_ok=True)
HYBRID_RESULT['reduced_universe'].screening_table.to_csv(HYBRID_OUTPUT / 'independent_marginal_utility_qubit_universe.csv')
model = HYBRID_RESULT['selection_model']
pd.DataFrame(model.Q, index=model.tickers, columns=model.tickers).to_csv(HYBRID_OUTPUT / 'independent_active_selection_qubo_matrix.csv')
pd.Series(model.linear, index=model.tickers, name='linear_coefficient').to_csv(HYBRID_OUTPUT / 'independent_active_selection_qubo_linear.csv')
HYBRID_RESULT['exact_enumeration'].to_csv(HYBRID_OUTPUT / 'independent_exact_qubo_enumeration.csv', index=False)
HYBRID_RESULT['qiskit_run']['samples'].to_csv(HYBRID_OUTPUT / 'independent_qaoa_samples.csv', index=False)
HYBRID_RESULT['qiskit_run']['callback_history'].to_csv(HYBRID_OUTPUT / 'independent_qaoa_convergence.csv', index=False)
HYBRID_RESULT['refinement_table'].to_csv(HYBRID_OUTPUT / 'independent_active_set_refinements.csv', index=False)
QAOA_SEED_SUMMARY.to_csv(HYBRID_OUTPUT / 'qaoa_seed_reliability.csv', index=False)
QAOA_RELIABILITY_SUMMARY.to_csv(HYBRID_OUTPUT / 'qaoa_reliability_summary.csv')
TARGET_RECOVERY_REDUCED.screening_table.to_csv(HYBRID_OUTPUT / 'target_recovery_qubit_universe.csv')
TARGET_RECOVERY_ENUMERATION.to_csv(HYBRID_OUTPUT / 'target_recovery_exact_qubo_enumeration.csv', index=False)
TARGET_RECOVERY_AUDIT.to_csv(HYBRID_OUTPUT / 'target_recovery_audit.csv')
TARGET_RECOVERY_EXACT_PROFILE['result'].weights.rename('weight').to_csv(HYBRID_OUTPUT / 'target_recovery_exact_weights.csv')
CLASSICAL_SUBSET_BASELINE_TABLE.to_csv(HYBRID_OUTPUT / 'classical_subset_baseline_candidates.csv', index=False)
CLASSICAL_BASELINE_SUMMARY.to_csv(HYBRID_OUTPUT / 'classical_subset_baseline_summary.csv', index=False)
comparison.to_csv(HYBRID_OUTPUT / 'finance_profile_comparison.csv')
_trade_table.to_csv(HYBRID_OUTPUT / 'active_candidates_and_executed_trades.csv')
hybrid_audit.to_csv(HYBRID_OUTPUT / 'hard_constraint_audit.csv', index=False)
scenario_comparison.to_csv(HYBRID_OUTPUT / 'scenario_warning_and_hard_limit_audit.csv')
PRIMARY_CLASSICAL_REFERENCE['result'].weights.rename('weight').to_csv(HYBRID_OUTPUT / 'primary_classical_weights.csv')
HYBRID_QAOA_PROFILE_RESULT['result'].weights.rename('weight').to_csv(HYBRID_OUTPUT / 'independent_qaoa_weights.csv')
if EXACT_ACTIVE_PROFILE_RESULT is not None:
    EXACT_ACTIVE_PROFILE_RESULT['result'].weights.rename('weight').to_csv(HYBRID_OUTPUT / 'independent_exact_active_set_weights.csv')
if STRICT_WARNING_REFERENCE is not None:
    STRICT_WARNING_REFERENCE['result'].weights.rename('weight').to_csv(HYBRID_OUTPUT / 'strict_warning_classical_weights.csv')
if not INSTITUTIONAL_SENSITIVITY_TABLE.empty:
    INSTITUTIONAL_SENSITIVITY_TABLE.to_csv(HYBRID_OUTPUT / 'institutional_cost_sensitivity.csv', index=False)
if not FORWARD_SIMULATION_SUMMARY.empty:
    FORWARD_SIMULATION_SUMMARY.to_csv(HYBRID_OUTPUT / 'forward_simulation_summary.csv')
    FORWARD_SIMULATION_RAW.to_csv(HYBRID_OUTPUT / 'forward_simulation_paths.csv', index=False)

def _installed_version(distribution_name: str) -> str:
    try:
        return importlib.metadata.version(distribution_name)
    except importlib.metadata.PackageNotFoundError:
        return 'not-installed'
classical_objective = float(comparison.loc['Primary unrestricted classical', 'normalized_objective'])
qaoa_objective = float(comparison.loc['Independent Qiskit QAOA', 'normalized_objective'])
exact_objective = float(comparison.loc['Independent exact active-set benchmark', 'normalized_objective']) if 'Independent exact active-set benchmark' in comparison.index else None
metadata = {'method_version': 'hybrid_qaoa', 'data_source': DATA_SOURCE, 'evaluation_type': 'historical' if DATA_SOURCE == 'yfinance' else 'synthetic_in_sample', 'synthetic_results_are_not_historical_backtests': DATA_SOURCE == 'synthetic', 'risk_policy_mode': RISK_POLICY_MODE, 'cost_scenario': STEP4_COST_SCENARIO, 'fast_mode': bool(FAST_MODE), 'selection_experiments': {'primary': 'incumbent_marginal_utility', 'separate_audit': 'classical_target_recovery'}, 'preferences': {'growth': GROWTH_SCORE, 'income': INCOME_SCORE, 'drawdown_control': DRAWDOWN_SCORE, 'cost_sensitivity': COST_SCORE}, 'qiskit_versions': {'qiskit': _installed_version('qiskit'), 'qiskit_algorithms': _installed_version('qiskit-algorithms'), 'qiskit_optimization': _installed_version('qiskit-optimization'), 'qiskit_aer': _installed_version('qiskit-aer')}, 'qaoa': {'seed_count': int(len(QAOA_SEED_SUMMARY)), 'reps': int(HYBRID_CONFIG.reps), 'shots': int(HYBRID_CONFIG.shots), 'maxiter': int(HYBRID_CONFIG.maxiter), 'cardinality': int(HYBRID_RESULT['cardinality']), 'candidate_qubits': int(HYBRID_RESULT['reduced_universe'].n_qubits), 'exact_optimum_hit_rate': float(QAOA_RELIABILITY_SUMMARY['exact_optimum_hit_rate']), 'median_energy_gap': float(QAOA_RELIABILITY_SUMMARY['median_qubo_energy_gap']), 'worst_energy_gap': float(QAOA_RELIABILITY_SUMMARY['worst_qubo_energy_gap']), 'mixer_type': HYBRID_RESULT['qiskit_run']['mixer_type'], 'aggregation': HYBRID_RESULT['qiskit_run']['aggregation']}, 'objectives': {'primary_classical': classical_objective, 'independent_qaoa': qaoa_objective, 'independent_exact_active_set': exact_objective, 'qaoa_gap_to_classical': qaoa_objective - classical_objective}, 'constraint_reporting': {'all_hard_constraints_satisfied': bool(hybrid_audit['satisfied'].all() and scenario_comparison['hard_limit_satisfied'].all()), 'soft_warning_breaches': int((~scenario_comparison['warning_satisfied']).sum())}, 'institutional_cost_sensitivity_generated': bool(not INSTITUTIONAL_SENSITIVITY_TABLE.empty), 'forward_simulation_generated': bool(not FORWARD_SIMULATION_SUMMARY.empty), 'claims_excluded': ['quantum advantage', 'quantum speedup', 'historical performance from synthetic data', 'out-of-sample market performance from synthetic data']}
metadata_path = HYBRID_OUTPUT / 'run_metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2, default=str), encoding='utf-8')
json.loads(metadata_path.read_text(encoding='utf-8'))
required_outputs = [HYBRID_OUTPUT / 'independent_qaoa_samples.csv', HYBRID_OUTPUT / 'independent_exact_qubo_enumeration.csv', HYBRID_OUTPUT / 'target_recovery_audit.csv', HYBRID_OUTPUT / 'classical_subset_baseline_summary.csv', HYBRID_OUTPUT / 'hard_constraint_audit.csv', metadata_path]
missing_outputs = [str(path) for path in required_outputs if not path.exists()]
if missing_outputs:
    raise FileNotFoundError('Missing required export outputs: ' + ', '.join(missing_outputs))
complete_step3_step4 = bool(STEP3_OUTPUT.exists() and STEP4_OUTPUT.exists())
if complete_step3_step4:
    archive_base = Path('/content') / f'step3_step4_step_05q_hybrid_qaoa_{DATA_SOURCE}_{STEP4_COST_SCENARIO}'
    archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=OUTPUT_ROOT)
    print('Created complete Step 3–4–5Q package:', archive_path)
else:
    archive_base = Path('/content') / f'step_05q_hybrid_qaoa_{DATA_SOURCE}_{STEP4_COST_SCENARIO}'
    archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=HYBRID_OUTPUT.parent, base_dir=HYBRID_OUTPUT.name)
    print('Step 3/4 output directories were not present in this restored runtime. Created accurately named Step 5Q package:', archive_path)
print('Archive size:', f'{Path(archive_path).stat().st_size / 1024 ** 2:.2f} MB')
files.download(archive_path)


Created complete Step 3–4–5Q package: /content/portfolio_pipeline_hybrid_qaoa_synthetic_base.zip
Archive size: 1.67 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Resume procedure without Google Drive